<a href="https://colab.research.google.com/github/Don-Youssef/EyeOfAI-System/blob/main/EyeOfAI_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup and Dependency Installation

In [ ]:
import os
if not os.path.exists('detectron2'):
  !git clone 'https://github.com/facebookresearch/detectron2'
import distutils.core
distutils = distutils.core.run_setup("./detectron2/setup.py")
!python -m pip install {' '.join([f"'{x}'" for x in distutils.install_requires])}
import sys
sys.path.insert(0, os.path.abspath('./detectron2'))
if not os.path.exists('Detic'):
  !git clone https://github.com/facebookresearch/Detic.git --recurse-submodules
%cd Detic
!pip install -r requirements.txt
import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()
from detectron2 import model_zoo
from detectron2.config import get_cfg
sys.path.insert(0, 'third_party/CenterNet2/')
from centernet.config import add_centernet_config
from detic.config import add_detic_config
from detectron2.engine import DefaultPredictor
from detectron2.data import MetadataCatalog
from detic.modeling.utils import reset_cls_test
!pip install 'torch==2.3.0'
!pip install 'torchvision==0.18.0'

#System Architecture Core

In [ ]:
# ============================================================ #
# BOOTSTRAP
# ============================================================ #
import subprocess, sys, os, pathlib  # Import standard libraries for subprocess execution, system operations, OS interaction, and path handling
import numpy as np  # Import NumPy for numerical array operations
import torch  # Import PyTorch for deep learning operations

# Determine PyTorch installation directory and prepare a serialization shim
torch_dir = pathlib.Path(torch.__file__).parent  # Get the parent directory of the torch package installation
ser_path  = torch_dir / "utils" / "serialization.py"  # Build the path to the serialization module inside torch/utils
ser_path.parent.mkdir(parents=True, exist_ok=True)  # Create the parent directory for ser_path if it does not already exist

# Create a serialization shim if it does not exist to ensure compatibility
# with older PyTorch versions or specific library expectations.
if not ser_path.exists():  # Check if the serialization shim file is missing
    content = '''# Compatibility shim for torch.utils.serialization
import types
class _C:
    USE_LEGACY_FORMAT = False
    DEFAULT_PROTOCOL  = 2
config           = _C()
DEFAULT_PROTOCOL = 2
def check_serialization_settings(): pass
def load(*a, **kw): return types.SimpleNamespace()
'''  # Define the shim content as a string with a minimal compatibility stub
    with open(ser_path, 'w') as f:  # Open the shim file for writing
        f.write(content)  # Write the shim content to the file
    # Reload modules to pick up the new shim
    for key in list(sys.modules.keys()):  # Iterate over all currently loaded module keys
        if 'torch.utils.serialization' in key:  # Check if the module key relates to torch serialization
            del sys.modules[key]  # Unload the stale module so the new shim is used on next import

# Install rich library for enhanced console output
subprocess.run("pip install -q rich", shell=True, capture_output=True)  # Quietly install the rich library via pip
from rich.console import Console  # Import the Console class from rich for styled terminal output
_run_console = Console()  # Create a global rich Console instance for pipeline-wide logging

# Helper function to run shell commands and display rich output
def run(cmd, desc=""):  # Define a helper that runs a shell command and prints styled status
    _run_console.print(f"  [bold magenta]>[/bold magenta]  [cyan]{desc or cmd[:60]}[/cyan]")  # Print the command description in cyan
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)  # Execute the shell command and capture stdout/stderr
    if result.returncode != 0:  # Check if the command exited with a non-zero return code indicating failure
        _run_console.print(f"  [bold yellow]WARNING:[/bold yellow] {result.stderr[-400:]}")  # Print the last 400 characters of stderr as a warning
    else:  # Command succeeded
        _run_console.print(f"  [bold green]SUCCESS[/bold green]")  # Print a green SUCCESS message
    return result  # Return the completed process result object

# Display a welcome banner
_run_console.print("╔══════════════════════════════════════════════════════╗", style="bold red")  # Print the top border of the welcome banner
_run_console.print("║         EyeOfAI  BOOTSTRAPPING INTELLIGENCE CORE                                         ║", style="bold red")  # Print the banner title line
_run_console.print("╚══════════════════════════════════════════════════════╝", style="bold red")  # Print the bottom border of the welcome banner

# Create necessary directories for Detic models and datasets
os.makedirs("/content/Detic/datasets/metadata", exist_ok=True)  # Create the Detic metadata directory, ignoring errors if it already exists
os.makedirs("/content/Detic/models",            exist_ok=True)  # Create the Detic models directory, ignoring errors if it already exists

# Configure environment variables to force DeepFace (TensorFlow/Keras backend) to use CPU
# and suppress TensorFlow warnings.
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"  # Hide all CUDA devices from TensorFlow so it runs on CPU only
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # Suppress TensorFlow C++ log messages below WARNING level

# Patch numpy type aliases which were deprecated in NumPy 1.24+ for compatibility
for _k, _v in {'bool': bool, 'float': float, 'int': int, 'object': object}.items():  # Iterate over numpy alias names and their Python built-in equivalents
    if not hasattr(np, _k):  # Check if this alias is missing from the current NumPy version
        setattr(np, _k, _v)  # Restore the alias by mapping it to the corresponding Python built-in

# Add custom paths to sys.path for importing local libraries
sys.path.insert(0, "/content/detectron2")  # Prepend the local detectron2 source path so it takes priority over installed packages
sys.path.insert(0, "/content/Detic")  # Prepend the local Detic source path for importability
sys.path.insert(0, "/content/Detic/third_party/CenterNet2")  # Prepend CenterNet2 (a Detic dependency) to the module search path
sys.path.insert(0, "/content/ByteTrack")  # Prepend the local ByteTrack source path for importability

# Check CUDA availability
cuda_available = torch.cuda.is_available()  # Detect whether a CUDA-capable GPU is available on this machine

# Install core dependencies
run("pip install --upgrade pip setuptools wheel",                           "Upgrade core Python tools")  # Upgrade pip, setuptools, and wheel to their latest versions
run("apt-get update -qq && apt-get install -y build-essential python3-dev git -qq", "Install system build tools")  # Install system-level C compiler, Python headers, and git
run("pip install -q opencv-python-headless Pillow requests",               "Install image processing and network libraries")  # Install OpenCV (headless), Pillow, and requests
run("pip install -q --only-binary :all: pyyaml",                           "Install PyYAML (binary)")  # Install PyYAML using a pre-built binary wheel to avoid compilation
run("pip install -q cython",                                               "Install Cython")  # Install Cython which is needed to compile some C extensions
run("pip install -q matplotlib pycocotools",                               "Install plotting and COCO utilities")  # Install matplotlib for plotting and pycocotools for COCO dataset helpers
run("pip install -q ninja",                                                "Install Ninja build system")  # Install the Ninja build system for faster C++ compilation

# Install PyTorch and torchvision based on CUDA availability
if cuda_available:  # Branch for CUDA-capable machines
    run("pip install -q 'torch==2.3.0' 'torchvision==0.18.0' --index-url https://download.pytorch.org/whl/cu121",
        "Install PyTorch with CUDA support")  # Install PyTorch 2.3.0 and torchvision with CUDA 12.1 support
else:  # Branch for CPU-only machines
    run("pip install -q 'torch==2.3.0' 'torchvision==0.18.0' --index-url https://download.pytorch.org/whl/cpu",
        "Install PyTorch for CPU")  # Install CPU-only PyTorch 2.3.0 and matching torchvision

# Determine PyTorch version details
TORCH_VER         = torch.__version__.split("+")[0]  # Extract the base version string by stripping any local build suffix
TORCH_MAJOR_MINOR = ".".join(TORCH_VER.split(".")[:2])  # Build a "major.minor" version string (e.g. "2.3") for wheel URL construction
CUDA_TAG          = "cu121" if cuda_available else "cpu"  # Select the CUDA tag string for download URLs based on GPU availability

# ── Detectron2 Installation ──────────────────────────────────────────────────
_d2_installed = False  # Track whether Detectron2 was successfully installed via a pre-built wheel
if cuda_available:  # Only attempt wheel installation on CUDA machines
    # Attempt to install Detectron2 using pre-built wheels for CUDA
    d2_wheel_url  = (f"https://dl.fbaipublicfiles.com/detectron2/wheels/"
                     f"{CUDA_TAG}/torch{TORCH_MAJOR_MINOR}/index.html")  # Construct the Detectron2 wheel index URL for the current torch+CUDA combination
    _d2_result    = run(f"pip install -q detectron2 -f {d2_wheel_url}", "Install Detectron2 wheel (CUDA)")  # Attempt to install Detectron2 from the pre-built wheel index
    _d2_installed = (_d2_result.returncode == 0)  # Mark wheel installation as successful if the return code is zero

if not _d2_installed:  # Fall back to source build if wheel installation was skipped or failed
    # Build Detectron2 from source as a fallback
    _run_console.print("  [bold yellow]Building Detectron2 from source (~5 min first run)...[/bold yellow]")  # Warn the user that a slow source build is starting
    run("pip install -q 'setuptools>=65' 'wheel' 'cython' 'numpy' 'torch' 'torchvision'",
        "Install Detectron2 build prerequisites")  # Install all prerequisites needed to compile Detectron2 from source
    if not os.path.exists("/content/detectron2"):  # Check if the Detectron2 source tree is already cloned
        run("git clone https://github.com/facebookresearch/detectron2.git /content/detectron2",
            "Clone Detectron2 repository")  # Clone the Detectron2 repository from GitHub into /content/detectron2
    _d2_src = run("pip install -q /content/detectron2", "Install Detectron2 from source")  # Attempt a standard pip install from the local source directory
    if _d2_src.returncode != 0:  # Check if the standard install failed
        run("pip install -q --no-build-isolation /content/detectron2",
            "Install Detectron2 from source (no-build-isolation fallback)")  # Retry the install without build isolation to work around environment conflicts

# ── Detic Installation ───────────────────────────────────────────────────────
if not os.path.exists("/content/Detic"):  # Check if the Detic repository has not been cloned yet
    run("git clone https://github.com/facebookresearch/Detic.git /content/Detic --recursive", "Clone Detic repository")  # Clone Detic and its submodules recursively from GitHub
run("cd /content/Detic && pip install -q -r requirements.txt", "Install Detic requirements")  # Install all Python packages listed in Detic's requirements file

_DETIC_ROOT    = "/content/Detic"  # Root directory of the Detic installation
_META_DIR      = f"{_DETIC_ROOT}/datasets/metadata"  # Directory where Detic vocabulary classifier files are stored
_MODELS_DIR    = f"{_DETIC_ROOT}/models"  # Directory where Detic model weight files are stored
_CAT_FREQ_PATH = f"{_META_DIR}/lvis_v1_train_cat_info.json"  # Path to the LVIS category frequency JSON used by Detic

# Download Detic metadata files
_BUILDIN_CLASSIFIER = {
    "lvis":       f"{_META_DIR}/lvis_v1_clip_a+cname.npy",  # Local path for the LVIS CLIP classifier embedding file
    "objects365": f"{_META_DIR}/o365_clip_a+cnamefix.npy",  # Local path for the Objects365 CLIP classifier embedding file
    "openimages": f"{_META_DIR}/oid_clip_a+cname.npy",  # Local path for the OpenImages CLIP classifier embedding file
    "coco":       f"{_META_DIR}/coco_clip_a+cname.npy",  # Local path for the COCO CLIP classifier embedding file
}
_CLASSIFIER_URLS = {
    "lvis":       "https://dl.fbaipublicfiles.com/detic/lvis_v1_clip_a+cname.npy",  # Remote download URL for the LVIS CLIP classifier
    "objects365": "https://dl.fbaipublicfiles.com/detic/o365_clip_a+cnamefix.npy",  # Remote download URL for the Objects365 CLIP classifier
    "openimages": "https://dl.fbaipublicfiles.com/detic/oid_clip_a+cname.npy",  # Remote download URL for the OpenImages CLIP classifier
    "coco":       "https://dl.fbaipublicfiles.com/detic/coco_clip_a+cname.npy",  # Remote download URL for the COCO CLIP classifier
}

# Download each classifier using the correct destination path from _BUILDIN_CLASSIFIER
for _vocab_key, _url in _CLASSIFIER_URLS.items():  # Iterate over each vocabulary key and its corresponding download URL
    _fpath = _BUILDIN_CLASSIFIER[_vocab_key]  # Resolve the local destination path for this vocabulary classifier
    if not (os.path.exists(_fpath) and os.path.getsize(_fpath) > 0):  # Download only if the file is missing or empty
        run(f"wget -q '{_url}' -O '{_fpath}'", f"Download {_vocab_key} classifier")  # Download the classifier file quietly to the expected local path

# Generate lvis_v1_categories.py if not present
_lvis_cats_path = "/content/Detic/detic/data/datasets/lvis_v1_categories.py"  # Path where the auto-generated LVIS category file should reside
if not os.path.exists(_lvis_cats_path):  # Only generate the file if it does not already exist
    _run_console.print("  [bold blue]Generating lvis_v1_categories.py...[/bold blue]")  # Inform the user that category file generation is starting
    run("pip install -q lvis", "Install LVIS API for category generation")  # Install the LVIS Python API needed for category metadata
    _create_script = r'''
import json, os, urllib.request
os.makedirs("/content/Detic/detic/data/datasets", exist_ok=True)
cat_freq_path = "/content/Detic/datasets/metadata/lvis_v1_train_cat_info.json"
categories = []
if os.path.exists(cat_freq_path):
    with open(cat_freq_path) as f:
        cat_info = json.load(f)
    categories = [{"id": c.get("id", i+1), "name": c["name"]} for i, c in enumerate(cat_info)]
else:
    try:
        url = "https://dl.fbaipublicfiles.com/detic/lvis_v1_train_cat_info.json"
        with urllib.request.urlopen(url, timeout=30) as r:
            cat_info = json.loads(r.read())
        categories = [{"id": c.get("id", i+1), "name": c["name"]} for i, c in enumerate(cat_info)]
    except Exception as e:
        # Fallback categories if download fails
        categories = [{"id":1,"name":"person"},{"id":2,"name":"bicycle"},{"id":3,"name":"car"}]
out_path = "/content/Detic/detic/data/datasets/lvis_v1_categories.py"
with open(out_path, "w") as f:
    f.write("# Auto-generated LVIS categories\n")
    f.write("LVIS_CATEGORIES = " + repr(categories) + "\n")
print(f"Written {len(categories)} categories -> {out_path}")
'''  # Python script source that reads/downloads LVIS category info and writes the categories module
    with open("/tmp/create_lvis_cats.py", "w") as f:  # Open a temporary file for writing the generator script
        f.write(_create_script)  # Write the category generator script to the temporary file
    run("python /tmp/create_lvis_cats.py", "Execute script to generate lvis_v1_categories.py")  # Execute the generator script as a subprocess

# Create __init__.py files for Detic modules if they do not exist
for _init_path in [
    "/content/Detic/detic/__init__.py",  # Package init for the top-level detic module
    "/content/Detic/detic/data/__init__.py",  # Package init for the detic.data sub-package
    "/content/Detic/detic/data/datasets/__init__.py",  # Package init for the detic.data.datasets sub-package
]:
    if not os.path.exists(_init_path):  # Only create the __init__.py if it is missing
        open(_init_path, "w").close()  # Create an empty __init__.py to make the directory a Python package

# Install CenterNet2 if present within Detic's third_party
if os.path.exists("/content/Detic/third_party/CenterNet2"):  # Check if CenterNet2 was included as a Detic submodule
    _has_setup  = os.path.exists("/content/Detic/third_party/CenterNet2/setup.py")  # Check whether CenterNet2 has a setup.py
    _has_pyproj = os.path.exists("/content/Detic/third_party/CenterNet2/pyproject.toml")  # Check whether CenterNet2 uses a pyproject.toml instead
    if _has_setup or _has_pyproj:  # Install only if a valid build configuration file exists
        run("pip install -q -e /content/Detic/third_party/CenterNet2", "Install CenterNet2 in editable mode")  # Install CenterNet2 in editable mode so source changes are immediately reflected

# ── ByteTrack Installation ───────────────────────────────────────────────────
if not os.path.exists("/content/ByteTrack"):  # Check if the ByteTrack repository has not been cloned yet
    run("git clone https://github.com/ifzhang/ByteTrack.git /content/ByteTrack --recursive", "Clone ByteTrack repository")  # Clone ByteTrack and its submodules from GitHub
    run("cd /content/ByteTrack && pip install -q -r requirements.txt 2>/dev/null || true", "Install ByteTrack requirements")  # Install ByteTrack's dependencies, tolerating any failures
    run("cd /content/ByteTrack && python setup.py develop 2>/dev/null || true",            "Install ByteTrack using setup.py")  # Install ByteTrack in develop mode, tolerating any failures

# Install additional dependencies for ByteTrack
run("pip install -q loguru lap cython_bbox thop", "Install loguru, lap, cython_bbox, thop")  # Install logging, linear assignment, bounding-box utilities, and model profiling libraries

# Patch ByteTrack for numpy type alias compatibility
_bt_path = "/content/ByteTrack/yolox/tracker/byte_tracker.py"  # Path to the ByteTrack source file that uses deprecated numpy aliases
if os.path.exists(_bt_path):  # Only apply patches if the ByteTrack file exists
    run(f"sed -i 's/np.float/float/g' {_bt_path}", "Patch ByteTrack for numpy.float alias")  # Replace all occurrences of np.float (deprecated) with the built-in float
    run(f"sed -i 's/np.int/int/g'   {_bt_path}",  "Patch ByteTrack for numpy.int alias")  # Replace all occurrences of np.int (deprecated) with the built-in int

# ── DeepFace Installation ────────────────────────────────────────────────────
run("pip install -q deepface tf-keras python-jose", "Install DeepFace and its dependencies")  # Install DeepFace, its TF-Keras backend, and JWT utilities

# ── pytorchvideo Installation for Action Recognition ─────────────────────────
run("pip install -q pytorchvideo", "Install pytorchvideo for action recognition")  # Install the pytorchvideo library used to load the MViT model
run("pip install -q 'av>=9.0.0'", "Install PyAV for pytorchvideo video I/O")  # Install PyAV (FFmpeg bindings) required by pytorchvideo for video decoding

# ── MViT-B 16x4 Kinetics-700 weights download ────────────────────────────────
# The official pytorchvideo torch.hub 'pretrained=True' loads K400 weights (400 classes).
# The Kinetics-700 checkpoint is listed in the pytorchvideo MODEL_ZOO.md under
# "MViT-B 16x4 | Kinetics-700" and must be downloaded separately.
_MVIT_K700_DIR  = "/content/mvit_k700"  # Local directory where the MViT Kinetics-700 checkpoint will be stored
_MVIT_K700_PATH = f"{_MVIT_K700_DIR}/MVIT_B_16x4_K700.pyth"  # Full local path for the MViT-B 16x4 Kinetics-700 checkpoint file
os.makedirs(_MVIT_K700_DIR, exist_ok=True)  # Create the checkpoint directory if it does not already exist
if not os.path.exists(_MVIT_K700_PATH):  # Download the checkpoint only if it is not already present
    run(
        f"wget -q 'https://dl.fbaipublicfiles.com/pytorchvideo/model_zoo/kinetics/MVIT_B_16x4.pyth'"
        f" -O '{_MVIT_K700_PATH}'",
        "Download MViT-B 16x4 Kinetics-700 weights",
    )  # Download the MViT-B 16x4 Kinetics-700 checkpoint from the PyTorchVideo model zoo

# ── Detic model weights Download ─────────────────────────────────────────────
_model_path = f"{_MODELS_DIR}/Detic_LCOCOI21k_CLIP_SwinB_896b32_4x_ft4x_max-size.pth"  # Local path where the Detic Swin-B model weights will be saved
if not os.path.exists(_model_path):  # Download model weights only if they are not already present
    run(f"wget -q https://dl.fbaipublicfiles.com/detic/Detic_LCOCOI21k_CLIP_SwinB_896b32_4x_ft4x_max-size.pth"
        f" -O {_model_path}", "Download Detic model weights")  # Download the Detic LVIS+COCO+Objects365+ImageNet-21k Swin-B weights

_run_console.print("\n  [bold green]Bootstrap complete.[/bold green]")  # Announce that the bootstrap phase has finished successfully

# ============================================================ #
# IMPORTS AND CORE SETUP
# ============================================================ #
import sys, os, time, threading, queue, logging, traceback, gc, json, warnings  # Import standard library modules for system, threading, I/O, and debugging
import concurrent.futures, urllib.request  # Import concurrent execution and URL fetching utilities
import cv2  # Import OpenCV for image and video processing
import numpy as np  # Import NumPy for array operations
import torch  # Import PyTorch for tensor operations and model inference
import psutil  # Import psutil for reading CPU and RAM statistics
from collections import defaultdict, deque, Counter  # Import efficient container types from the standard library
from dataclasses import dataclass, field  # Import dataclass decorator and field helper for structured configuration
from typing import Optional, List, Dict, Tuple, Any  # Import type hint helpers for clearer function signatures
from datetime import datetime  # Import datetime for timestamp formatting

warnings.filterwarnings("ignore")  # Suppress all Python warnings to keep console output clean
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # Select "cuda" if a GPU is available, otherwise fall back to "cpu"
if DEVICE == "cuda":  # Apply GPU-specific performance optimisations when running on CUDA
    torch.backends.cudnn.benchmark        = True  # Enable cuDNN auto-tuner to find the fastest convolution algorithms
    torch.backends.cuda.matmul.allow_tf32 = True  # Allow TF32 arithmetic for matrix multiplications on Ampere+ GPUs
    torch.backends.cudnn.allow_tf32       = True  # Allow TF32 arithmetic inside cuDNN convolution kernels

_run_console.print(f"  [bold blue]Device : {DEVICE.upper()}[/bold blue]")  # Print the selected compute device
if DEVICE == "cuda":  # Print additional GPU details only when running on CUDA
    _run_console.print(f"  [bold blue]GPU    : {torch.cuda.get_device_name(0)}[/bold blue]")  # Print the name of the first CUDA GPU
    _run_console.print(
        f"  [bold blue]VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB[/bold blue]")  # Print total VRAM of the first CUDA GPU in gigabytes

# ============================================================ #
# CONFIGURATION
# ============================================================ #
@dataclass
class EyeConfig:  # Dataclass that centralises every tunable parameter for the pipeline
    detic_threshold: float = 0.0  # Minimum detection confidence score; set to 0.0 to allow all detections
    detic_model_path: str  = (f"{_DETIC_ROOT}/models/"
                               "Detic_LCOCOI21k_CLIP_SwinB_896b32_4x_ft4x_max-size.pth")  # Absolute path to the Detic Swin-B model weights file
    detic_config: str      = (f"{_DETIC_ROOT}/configs/"
                               "Detic_LCOCOI21k_CLIP_SwinB_896b32_4x_ft4x_max-size.yaml")  # Absolute path to the Detic Swin-B model YAML configuration file
    vocabulary: str        = "lvis"  # Vocabulary to use for open-vocabulary detection (lvis/objects365/openimages/coco)

    track_thresh: float = 0.5  # Minimum detection confidence to create or maintain a ByteTrack track
    track_buffer: int   = 30  # Number of frames a track can be unmatched before it is discarded
    match_thresh: float = 0.8  # IoU threshold used when associating detections to existing tracks
    min_box_area: float = 100.0  # Minimum bounding-box area in pixels; smaller boxes are discarded

    deepface_backend: str          = "retinaface"  # Face detection backend passed to DeepFace.analyze (retinaface is most accurate)
    emotion_interval_sec: float    = 2.0  # Minimum seconds between successive emotion analyses for the same track
    age_gender_interval_sec: float = 4.0  # Minimum seconds between successive age/gender analyses for the same track
    deepface_frame_skip: int       = 10  # Analyse face attributes only on every N-th frame to reduce CPU load
    deepface_emotion_smooth_window: int = 5  # Number of recent emotion predictions to majority-vote over for smoothing
    deepface_emotion_confidence_threshold: float = 0.6  # Minimum dominant-emotion probability required to accept a prediction

    box_color: Tuple[int, int, int] = (0, 0, 220)  # BGR colour of the bounding-box border drawn around detected objects
    box_thickness: int              = 1  # Pixel thickness of the bounding-box border lines
    font_scale: float               = 0.42  # Scale factor applied to the OpenCV font when drawing label text
    l_size: int                     = 14  # Length of each arm of the L-shaped corner markers drawn on bounding boxes
    show_hud: bool                  = False  # Whether to overlay the performance HUD on each output frame

    input_width: int      = 960  # Width in pixels to which each frame is resized before processing
    input_height: int     = 540  # Height in pixels to which each frame is resized before processing
    max_fps: int          = 30  # Target maximum frames per second for video capture and processing
    frame_queue_size: int = 8  # Maximum number of frames buffered between the capture and processing threads

    gov_target_fps: float   = 25.0  # FPS target the Performance Governor tries to maintain
    gov_window: int         = 30  # Number of recent FPS samples used to compute the rolling average for the governor
    gov_check_interval: int = 10  # How many frames elapse between successive governor level evaluations
    gov_levels: int         = 3  # Total number of governor quality levels (0=full, 1=reduced, 2=lite)

    clahe_clip_init: float      = 2.0  # Initial CLAHE clip limit used by the Scene Calibrator
    clahe_tile: int             = 8  # Tile grid size (N×N) used when creating CLAHE objects
    brightness_ema_alpha: float = 0.1  # EMA smoothing factor applied to the frame brightness estimate
    bright_lo: float            = 60.0  # Brightness below this threshold triggers aggressive CLAHE enhancement
    bright_hi: float            = 190.0  # Brightness above this threshold triggers mild CLAHE to avoid over-exposure

    async_workers: int = 2  # Number of background worker threads used by the AsyncDeepFacePool

    # Action recognition settings
    # MViT-B 16x4 name means 16 temporal frames x stride 4 — these MUST match exactly.
    action_clip_len: int         = 16  # Number of temporal frames fed to MViT (must be 16 to match positional embeddings)
    action_frame_stride: int     = 4   # Temporal stride between sampled frames (must be 4 to match training setup)
    action_interval_frames: int  = 16  # How many new frames must arrive before triggering another action inference
    action_top_k: int            = 1   # Number of top-ranked action predictions to store and display
    # Path to the manually downloaded MViT-B 16x4 Kinetics-700 checkpoint.
    # torch.hub pretrained=True loads K400 (400 classes); this path gives K700 (700 classes).
    mvit_k700_path: str          = "/content/mvit_k700/MVIT_B_16x4_K700.pyth"  # Local filesystem path for the Kinetics-700 MViT checkpoint downloaded during bootstrap

    log_level: str = "INFO"  # Python logging level (DEBUG/INFO/WARNING/ERROR) for the pipeline logger
    log_file: str  = "/content/eyeofai_debug.log"  # Path to the file where all pipeline log messages are appended

CFG = EyeConfig()  # Instantiate the global configuration object with default values
_run_console.print("  [bold green]Configuration loaded.[/bold green]")  # Confirm that the configuration dataclass was created successfully

# ============================================================ #
# LOGGING AND MONITORING
# ============================================================ #
from rich.logging import RichHandler  # Import the rich log handler for styled terminal log output
from rich.panel   import Panel  # Import Panel for rendering bordered text boxes in the terminal
from rich.console import Console as _RichConsole  # Import Console under an alias to avoid name collision

# Initialize rich console for logging
_console = _RichConsole()  # Create a dedicated rich Console instance used exclusively by the logger
logging.basicConfig(
    level=getattr(logging, CFG.log_level),  # Set the root log level from the configuration
    format="%(message)s", datefmt="[%X]",  # Use a minimal log format with HH:MM:SS timestamps
    handlers=[
        RichHandler(console=_console, rich_tracebacks=True, markup=True),  # Add a rich-styled console handler with traceback support
        logging.FileHandler(CFG.log_file, mode="a"),  # Append all log messages to the configured log file
    ],
)
log = logging.getLogger("EyeOfAI")  # Create a named logger for all EyeOfAI pipeline messages

# SystemMonitor class tracks performance metrics and logs events
class SystemMonitor:
    def __init__(self):
        self.fps_history: deque      = deque(maxlen=60)  # Rolling window of the last 60 FPS measurements
        self.detection_count: deque  = deque(maxlen=60)  # Rolling window of the last 60 per-frame detection counts
        self.error_log: List[str]    = []  # Accumulates all ERROR-level log entries for later review
        self.warnings_log: List[str] = []  # Accumulates all WARNING-level log entries for later review
        self.events_log: deque       = deque(maxlen=200)  # Circular buffer holding the 200 most recent event strings
        self._lock                   = threading.Lock()  # Thread lock protecting all mutable state in this class
        self._start                  = time.time()  # Wall-clock timestamp recorded when the monitor was created
        self.frame_count             = 0  # Total number of frames processed since the pipeline started
        self.total_detections        = 0  # Cumulative count of all object detections across all frames
        self.governor: Optional[Any] = None  # Reference to the PerformanceGovernor set after it is created

    def push_fps(self, fps):
        """Records the frames per second for averaging."""
        with self._lock: self.fps_history.append(fps)  # Thread-safely append the current FPS sample to the history buffer

    def push_detection(self, n):
        """Records the number of detections and updates total count."""
        with self._lock:
            self.detection_count.append(n)  # Append the per-frame detection count to the rolling window
            self.total_detections += n  # Increment the cumulative detection counter

    def log_event(self, msg, level="INFO"):
        """Logs an event with a timestamp and level."""
        ts    = datetime.now().strftime("%H:%M:%S.%f")[:-3]  # Format the current time as HH:MM:SS.mmm (millisecond precision)
        entry = f"[{ts}] [{level}] {msg}"  # Build the full log entry string with timestamp and level prefix
        with self._lock:
            self.events_log.appendleft(entry)  # Prepend the entry so the most recent event appears first
            if level == "ERROR": self.error_log.append(entry)  # Also store errors in the dedicated error list
            elif level == "WARN": self.warnings_log.append(entry)  # Also store warnings in the dedicated warnings list

    def avg_fps(self):
        """Calculates the average FPS over the history."""
        return float(np.mean(self.fps_history)) if self.fps_history else 0.0  # Return the mean of stored FPS samples, or 0 if the buffer is empty

    def gpu_stats(self):
        """Retrieves current GPU memory usage if CUDA is available."""
        if DEVICE == "cuda":
            used  = torch.cuda.memory_allocated() / 1e9  # Convert allocated GPU memory from bytes to gigabytes
            total = torch.cuda.get_device_properties(0).total_memory / 1e9  # Convert total GPU memory from bytes to gigabytes
            return {"used_gb": round(used, 2), "total_gb": round(total, 2),
                    "pct": round(used / total * 100, 1)}  # Return a dict with used, total, and percentage GPU memory
        return {"used_gb": 0, "total_gb": 0, "pct": 0}  # Return zeros when no GPU is present

    def cpu_stats(self):
        """Retrieves current CPU and RAM usage."""
        return {"pct": psutil.cpu_percent(),  # Percentage of CPU currently in use
                "ram_gb": round(psutil.virtual_memory().used / 1e9, 2),  # Used RAM in gigabytes
                "ram_total_gb": round(psutil.virtual_memory().total / 1e9, 2)}  # Total installed RAM in gigabytes

    def uptime(self):
        """Calculates the system uptime."""
        s = int(time.time() - self._start)  # Total elapsed seconds since the monitor was initialised
        return f"{s//3600:02d}:{(s%3600)//60:02d}:{s%60:02d}"  # Format elapsed time as HH:MM:SS

MONITOR = SystemMonitor()  # Create the global SystemMonitor singleton used across the whole pipeline
_run_console.print("  [bold green]System Monitor online.[/bold green]")  # Confirm that the system monitor has been initialised

def self_heal(func_name, exception, retry_fn=None, max_retries=3):
    """
    Attempts to self-heal from exceptions by logging and optionally retrying.
    Clears CUDA cache and collects garbage on failure.
    """
    msg = f"FAULT in [{func_name}]: {type(exception).__name__}: {str(exception)[:120]}"  # Build a concise fault message with the exception type and a truncated message
    log.error(msg); MONITOR.log_event(msg, "ERROR")  # Log the fault to both the Python logger and the SystemMonitor event log
    if retry_fn and max_retries > 0:  # Check if a retry callback was provided and retries remain
        log.warning(f"  Self-heal: retrying [{func_name}] ({max_retries} left)...")  # Log the retry attempt
        time.sleep(0.2)  # Brief pause before retrying to allow transient conditions to resolve
        try:    return retry_fn(), True  # Execute the retry callback and signal success
        except Exception as e2: return self_heal(func_name, e2, retry_fn, max_retries - 1)  # Recursively attempt self-healing with one fewer retry
    if DEVICE == "cuda": torch.cuda.empty_cache()  # Release cached GPU memory to recover from potential OOM conditions
    gc.collect()  # Run garbage collection to reclaim unreferenced Python objects
    return None, False  # Return None result and False success flag after exhausting all retries

# ============================================================ #
# PERFORMANCE GOVERNOR
# Dynamically adjusts processing complexity based on system FPS.
# ============================================================ #
class PerformanceGovernor:
    _LEVEL_NAMES = ["FULL", "REDUCED_FACE", "LITE"]  # Human-readable names for each governor quality level

    def __init__(self, cfg, monitor):
        self.cfg    = cfg  # Store a reference to the global EyeConfig for threshold and FPS parameters
        self.monitor = monitor  # Store a reference to the SystemMonitor for event logging
        self.level  = 0  # Start at FULL quality (level 0) and degrade only when FPS falls behind
        self._frame_idx       = 0  # Counter incremented on every tick() call to track evaluation timing
        self._fps_buf: deque  = deque(maxlen=cfg.gov_window)  # Rolling FPS buffer sized to the configured governor window
        self._lock            = threading.Lock()  # Thread lock protecting governor state from concurrent access
        self._recover_patience = 0  # Counter tracking how many consecutive intervals have had good FPS before recovering
        self._RECOVER_NEEDED   = 20  # Number of check intervals at good FPS required before stepping up quality
        _run_console.print(
            f"  [bold green]Performance Governor online | target={cfg.gov_target_fps} FPS[/bold green]")  # Announce that the governor is active with its target FPS

    @property
    def level_name(self):
        """Returns the human-readable name of the current governor level."""
        return self._LEVEL_NAMES[self.level]  # Look up and return the name string for the current level index

    def tick(self, fps):
        """Updates the governor based on current FPS and adjusts level."""
        with self._lock:
            self._fps_buf.append(fps)  # Add the latest FPS sample to the rolling buffer
            self._frame_idx += 1  # Advance the frame counter
            if self._frame_idx % self.cfg.gov_check_interval != 0: return self.level  # Skip evaluation on non-check frames to reduce overhead
            if len(self._fps_buf) < self.cfg.gov_window // 2:       return self.level  # Wait until the buffer has enough samples for a reliable average
            avg         = float(np.mean(self._fps_buf))  # Compute the rolling average FPS from the buffer
            load_factor = self.cfg.gov_target_fps / max(avg, 0.1)  # Compute ratio of target FPS to actual FPS; values above 1.0 mean the system is overloaded
            prev        = self.level  # Remember the current level before any change
            if load_factor > 1.35 and self.level < self.cfg.gov_levels - 1:  # Degrade if actual FPS is 35% below target and we are not already at the lowest quality
                self.level = min(self.level + 1, self.cfg.gov_levels - 1)  # Step down one quality level
                self._recover_patience = 0  # Reset recovery patience since we just degraded
                self.monitor.log_event(f"[GOV] Degraded L{prev}→L{self.level} ({avg:.1f}fps)", "WARN")  # Log the quality degradation event
            elif load_factor < 0.80 and self.level > 0:  # Consider recovering if actual FPS is 20% above target and quality has been reduced
                self._recover_patience += self.cfg.gov_check_interval  # Accumulate patience toward the recovery threshold
                if self._recover_patience >= self._RECOVER_NEEDED:  # Recover only after sustained good performance
                    self.level = max(self.level - 1, 0)  # Step up one quality level
                    self._recover_patience = 0  # Reset patience counter after recovering
                    self.monitor.log_event(f"[GOV] Recovered L{prev}→L{self.level} ({avg:.1f}fps)", "INFO")  # Log the quality recovery event
            else:
                self._recover_patience = 0  # Reset patience when neither degrading nor recovering
            return self.level  # Return the (possibly updated) governor level

    def should_run_face(self, base_interval):
        """Adjusts DeepFace analysis interval based on governor level."""
        return base_interval * [1, 2, 4][self.level]  # Multiply the base interval by 1/2/4 depending on quality level

    def effective_resolution(self, w, h):
        """Adjusts frame resolution based on governor level."""
        if self.level == 0:
            return (w, h)  # FULL quality: use the configured resolution unchanged
        elif self.level == 1:
            return (int(w * 0.75), int(h * 0.75))  # REDUCED_FACE quality: scale down to 75% of configured resolution
        else:
            return (w // 2, h // 2)  # LITE quality: halve both dimensions to significantly reduce computation

    def effective_threshold(self, base):
        """Adjusts detection threshold based on governor level."""
        return min(base + [0.0, 0.05, 0.10][self.level], 0.90)  # Raise the confidence threshold at lower quality levels to skip uncertain detections

    def target_frame_duration(self):
        """Calculates target frame processing duration to meet FPS goal."""
        return [1.0, 1.1, 1.25][self.level] / self.cfg.gov_target_fps  # Compute allowed seconds per frame, relaxed slightly at lower quality levels

# ============================================================ #
# ADAPTIVE SCENE CALIBRATION
# Adjusts image brightness/contrast using CLAHE.
# ============================================================ #
class SceneCalibrator:
    def __init__(self, cfg):
        self.cfg          = cfg  # Store a reference to EyeConfig for brightness thresholds and CLAHE parameters
        self._clahe       = cv2.createCLAHE(clipLimit=cfg.clahe_clip_init,
                                             tileGridSize=(cfg.clahe_tile, cfg.clahe_tile))  # Create the initial CLAHE object with configured clip limit and tile grid size
        self._ema_bright  = 128.0  # Initialise the EMA brightness estimate to a neutral mid-grey value
        self._ema_clip    = cfg.clahe_clip_init  # Initialise the EMA CLAHE clip limit to the configured starting value
        self._conf_offset = 0.0  # Detection confidence offset applied based on scene brightness conditions
        self._alpha       = cfg.brightness_ema_alpha  # EMA smoothing coefficient for brightness and clip limit updates
        _run_console.print("  [bold green]Scene Calibrator online.[/bold green]")  # Confirm that the scene calibrator has been initialised

    def _ema(self, new, old):
        """Performs Exponential Moving Average calculation."""
        return self._alpha * new + (1.0 - self._alpha) * old  # Blend new observation with previous EMA using the configured alpha

    def _brightness(self, frame):
        """Calculates the average brightness of a downscaled frame."""
        return float(np.mean(cv2.cvtColor(cv2.resize(frame, (160, 90)), cv2.COLOR_BGR2HSV)[:, :, 2]))  # Downscale the frame to 160×90, convert to HSV, and average the V (value) channel

    def calibrate(self, frame):
        """Applies adaptive CLAHE based on frame brightness."""
        b = self._brightness(frame)  # Compute the current frame's average brightness
        self._ema_bright = self._ema(b, self._ema_bright)  # Update the smoothed brightness estimate with the new sample

        if self._ema_bright < self.cfg.bright_lo:  # Frame is too dark — boost CLAHE aggressiveness and lower detection confidence
            target_clip, self._conf_offset = 3.0, -0.07
        elif self._ema_bright > self.cfg.bright_hi:  # Frame is too bright — reduce CLAHE and slightly raise detection confidence
            target_clip, self._conf_offset = 1.2,  0.06
        else:  # Frame brightness is in the normal range — interpolate CLAHE linearly and reset the offset
            r = (self._ema_bright - self.cfg.bright_lo) / (self.cfg.bright_hi - self.cfg.bright_lo)  # Normalised position within the acceptable brightness range [0, 1]
            target_clip, self._conf_offset = 3.0 - r * 1.8, 0.0  # Linearly interpolate CLAHE clip between 3.0 (dark) and 1.2 (bright)

        self._ema_clip = self._ema(target_clip, self._ema_clip)  # Smoothly update the CLAHE clip limit toward the target value
        self._clahe    = cv2.createCLAHE(clipLimit=round(self._ema_clip, 2),
                                          tileGridSize=(self.cfg.clahe_tile, self.cfg.clahe_tile))  # Recreate the CLAHE object with the updated clip limit
        lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)  # Convert the frame from BGR to LAB colour space for luminance processing
        l, a, b_ = cv2.split(lab)  # Split the LAB image into its three channels
        return cv2.cvtColor(cv2.merge([self._clahe.apply(l), a, b_]), cv2.COLOR_LAB2BGR)  # Apply CLAHE to the L channel only, then merge and convert back to BGR

    @property
    def conf_offset(self):
        """Returns the current confidence offset for detection."""
        return self._conf_offset  # Expose the brightness-driven confidence offset to the detection pipeline

    @property
    def brightness(self):
        """Returns the exponentially averaged brightness."""
        return round(self._ema_bright, 1)  # Return the smoothed brightness estimate rounded to one decimal place

    @property
    def clahe_clip(self):
        """Returns the exponentially averaged CLAHE clip limit."""
        return round(self._ema_clip,   2)  # Return the smoothed CLAHE clip limit rounded to two decimal places

# ============================================================ #
# PERSON ATTRIBUTE STORE
# Stores emotion, age, gender, action per track_id.
# Each attribute has an independent refresh interval.
# get() always returns the last known value — never blocks.
# ============================================================ #
class PersonAttributeStore:
    def __init__(self, cfg):
        self.cfg   = cfg  # Store EyeConfig reference for refresh intervals and smoothing parameters
        self._lock = threading.Lock()  # Thread lock protecting all attribute dictionaries from race conditions

        self._emotion: Dict[int, str]       = {}  # Maps track ID to the latest smoothed emotion label
        self._emotion_ts: Dict[int, float]  = {}  # Maps track ID to the wall-clock time of the last emotion update
        self._emotion_hist: Dict[int, deque] = defaultdict(lambda: deque(maxlen=cfg.deepface_emotion_smooth_window))  # Smoothing buffer holding the last N emotion predictions per track
        self._analyzing_emotion: set        = set()  # Set of track IDs currently undergoing emotion analysis

        self._age: Dict[int, str]           = {}  # Maps track ID to the latest age range string
        self._age_ts: Dict[int, float]      = {}  # Maps track ID to the wall-clock time of the last age update
        self._analyzing_age: set            = set()  # Set of track IDs currently undergoing age analysis

        self._gender: Dict[int, str]        = {}  # Maps track ID to the latest gender label string
        self._gender_ts: Dict[int, float]   = {}  # Maps track ID to the wall-clock time of the last gender update
        self._analyzing_gender: set         = set()  # Set of track IDs currently undergoing gender analysis

        self._action: Dict[int, str]        = {}  # Maps track ID to the latest action recognition label

    def _needs(self, ts_dict, tid, interval):
        """Checks if an attribute needs an update based on its interval."""
        return (time.time() - ts_dict.get(tid, 0.0)) >= interval  # Return True if the elapsed time since the last update meets or exceeds the required interval

    def emotion_needs_update(self, tid):
        """Checks if emotion attribute needs an update for a given track ID."""
        with self._lock:
            return self._needs(self._emotion_ts, tid,
                               MONITOR.governor.should_run_face(self.cfg.emotion_interval_sec))  # Delegate to _needs using the governor-adjusted emotion interval

    def is_analyzing_emotion(self, tid):
        """Checks if emotion analysis is currently in progress for a track ID."""
        with self._lock: return tid in self._analyzing_emotion  # Return True if the track ID is in the in-progress set

    def mark_analyzing_emotion(self, tid):
        """Marks a track ID as currently undergoing emotion analysis."""
        with self._lock: self._analyzing_emotion.add(tid)  # Add the track ID to the in-progress set to prevent duplicate submissions

    def put_emotion(self, tid, v):
        """Stores the result of emotion analysis, applies smoothing."""
        with self._lock:
            self._emotion_hist[tid].append(v)  # Append the new emotion prediction to this track's smoothing buffer
            smoothed_emotion = Counter(self._emotion_hist[tid]).most_common(1)[0][0]  # Select the most frequently predicted emotion over the smoothing window
            self._emotion[tid]    = smoothed_emotion  # Store the smoothed emotion label for this track
            self._emotion_ts[tid] = time.time()  # Record the current time as the last update timestamp
            self._analyzing_emotion.discard(tid)  # Mark analysis as complete for this track ID

    def age_needs_update(self, tid):
        """Checks if age attribute needs an update for a given track ID."""
        with self._lock:
            return self._needs(self._age_ts, tid,
                               MONITOR.governor.should_run_face(self.cfg.age_gender_interval_sec))  # Delegate to _needs using the governor-adjusted age/gender interval

    def is_analyzing_age(self, tid):
        """Checks if age analysis is currently in progress for a track ID."""
        with self._lock: return tid in self._analyzing_age  # Return True if the track ID is in the in-progress set

    def mark_analyzing_age(self, tid):
        """Marks a track ID as currently undergoing age analysis."""
        with self._lock: self._analyzing_age.add(tid)  # Add the track ID to the in-progress set

    def put_age(self, tid, v):
        """Stores the result of age analysis."""
        with self._lock:
            self._age[tid]    = str(v)  # Convert the age value to a string and store it for this track
            self._age_ts[tid] = time.time()  # Record the current time as the last update timestamp
            self._analyzing_age.discard(tid)  # Mark analysis as complete for this track ID

    def gender_needs_update(self, tid):
        """Checks if gender attribute needs an update for a given track ID."""
        with self._lock:
            return self._needs(self._gender_ts, tid,
                               MONITOR.governor.should_run_face(self.cfg.age_gender_interval_sec))  # Delegate to _needs using the governor-adjusted age/gender interval

    def is_analyzing_gender(self, tid):
        """Checks if gender analysis is currently in progress for a track ID."""
        with self._lock: return tid in self._analyzing_gender  # Return True if the track ID is in the in-progress set

    def mark_analyzing_gender(self, tid):
        """Marks a track ID as currently undergoing gender analysis."""
        with self._lock: self._analyzing_gender.add(tid)  # Add the track ID to the in-progress set

    def put_gender(self, tid, v):
        """Stores the result of gender analysis."""
        with self._lock:
            self._gender[tid]    = str(v)  # Convert the gender value to a string and store it for this track
            self._gender_ts[tid] = time.time()  # Record the current time as the last update timestamp
            self._analyzing_gender.discard(tid)  # Mark analysis as complete for this track ID

    def put_action(self, tid, action_name: str):
        """Stores the predicted action name for a given track ID."""
        with self._lock:
            self._action[tid] = action_name  # Store the action label string for this track

    def get(self, tid):
        """Retrieves all attributes for a given track ID."""
        with self._lock:
            return {
                "emotion": self._emotion.get(tid, "Unknown"),  # Return stored emotion or "Unknown" if not yet analysed
                "age":     self._age.get(tid, "Unknown"),  # Return stored age range or "Unknown" if not yet analysed
                "gender":  self._gender.get(tid, "Unknown"),  # Return stored gender or "Unknown" if not yet analysed
                "action":  self._action.get(tid, "Unknown"),  # Return stored action label or "Unknown" if not yet analysed
            }

    def purge_stale(self, active_ids: set):
        """Clears attributes for track IDs that are no longer active."""
        with self._lock:
            for tid in set(self._emotion.keys()) - active_ids:  # Identify emotion entries whose tracks are gone
                self._emotion.pop(tid, None); self._emotion_ts.pop(tid, None)  # Clear stored emotion and its timestamp
                self._emotion_hist.pop(tid, None)  # Clear the smoothing history buffer
                self._analyzing_emotion.discard(tid)  # Ensure the track is not left in the in-progress set
            for tid in set(self._age.keys()) - active_ids:  # Identify age entries whose tracks are gone
                self._age.pop(tid, None); self._age_ts.pop(tid, None)  # Clear stored age and its timestamp
                self._analyzing_age.discard(tid)  # Ensure the track is not left in the in-progress set
            for tid in set(self._gender.keys()) - active_ids:  # Identify gender entries whose tracks are gone
                self._gender.pop(tid, None); self._gender_ts.pop(tid, None)  # Clear stored gender and its timestamp
                self._analyzing_gender.discard(tid)  # Ensure the track is not left in the in-progress set
            for tid in set(self._action.keys()) - active_ids:  # Identify action entries whose tracks are gone
                self._action.pop(tid, None)  # Clear the stored action label

# ============================================================ #
# ACTION RECOGNIZER
# Uses pytorchvideo's MViT trained on Kinetics-700
# following the official pytorchvideo model hub API.
# ============================================================ #

# Official Kinetics-700 class names list (700 classes)
# We load these from pytorchvideo's bundled JSON resource
_KINETICS_700_LABELS: Optional[List[str]] = None  # Module-level cache for the Kinetics-700 label list; None until first loaded

def _load_kinetics700_labels() -> List[str]:
    """
    Loads Kinetics-700 class label names using the official
    pytorchvideo / torchvision resource path, with a URL fallback.
    Returns a list of 700 string class names sorted by class index.
    """
    global _KINETICS_700_LABELS  # Declare intent to update the module-level cache variable
    if _KINETICS_700_LABELS is not None:  # Return immediately if labels have already been loaded
        return _KINETICS_700_LABELS

    # Try pytorchvideo bundled JSON (official path used by the model hub)
    try:
        from pytorchvideo.data.kinetics import Kinetics  # Import the Kinetics dataset class (triggers pytorchvideo package loading)
        # pytorchvideo stores class names as a dict {name: idx}
        # We reconstruct the sorted list by index
        import importlib.resources as _ir  # Standard library resource accessor (not directly used here)
        import pkgutil  # Standard library utility for accessing package data files
        # Path used internally by pytorchvideo
        data = pkgutil.get_data("pytorchvideo", "data/kinetics/kinetics700_labels.json")  # Read the bundled Kinetics-700 JSON as raw bytes
        if data:
            mapping: Dict[str, int] = json.loads(data.decode())  # Decode bytes and parse the JSON into a {label: index} dict
            labels = [""] * len(mapping)  # Pre-allocate a list with one slot per class
            for name, idx in mapping.items():
                labels[idx] = name  # Place each label at its correct index position
            _KINETICS_700_LABELS = labels  # Cache the loaded labels
            return _KINETICS_700_LABELS
    except Exception:
        pass  # Silently fall through to the next loading strategy

    # Fallback: download from PyTorch Hub official JSON
    try:
        url = "https://dl.fbaipublicfiles.com/pyslowfast/dataset/class_names/kinetics_classnames.json"  # Official Kinetics class-names JSON hosted by FAIR
        with urllib.request.urlopen(url, timeout=15) as resp:
            raw: Dict[str, int] = json.loads(resp.read().decode())  # Download and parse the class-names JSON
        labels = [""] * len(raw)  # Pre-allocate one slot per class
        for name, idx in raw.items():
            if idx < len(labels):
                labels[idx] = name  # Place each label at its correct index, ignoring out-of-range entries
        _KINETICS_700_LABELS = labels  # Cache the downloaded labels
        return _KINETICS_700_LABELS
    except Exception:
        pass  # Silently fall through to the next loading strategy

    # Second fallback: use torch hub JSON shipped with torchvision
    try:
        import torchvision  # Import torchvision to locate its package directory
        tv_root = pathlib.Path(torchvision.__file__).parent  # Get the torchvision installation directory
        candidates = list(tv_root.rglob("kinetics*700*.json")) + list(tv_root.rglob("kinetics*label*.json"))  # Search recursively for any Kinetics label JSON files inside torchvision
        if candidates:
            with open(candidates[0]) as f:
                raw = json.load(f)  # Load the first matching JSON file found
            if isinstance(raw, dict):
                labels = [""] * len(raw)  # Pre-allocate one slot per class
                for name, idx in raw.items():
                    if isinstance(idx, int) and idx < len(labels):
                        labels[idx] = name  # Place each label at its correct index
                _KINETICS_700_LABELS = labels  # Cache the labels found in torchvision
                return _KINETICS_700_LABELS
    except Exception:
        pass  # Silently fall through to the ultimate fallback

    # Ultimate fallback: return empty list; action will show "Unknown"
    _run_console.print("[bold yellow]Could not load Kinetics-700 labels; actions will show as 'Unknown'.[/bold yellow]")  # Warn the user that label loading failed completely
    _KINETICS_700_LABELS = []  # Store an empty list so the check `if _KINETICS_700_LABELS is not None` passes
    return _KINETICS_700_LABELS


class ActionRecognizer:
    """
    Wraps the official PyTorchVideo MViT-B 16x4 model trained on Kinetics-700.
    Follows the exact preprocessing pipeline described in the PyTorchVideo
    model hub documentation:
      - ShortSideScale to 256
      - CenterCrop to 224×224
      - Normalize with official Kinetics mean/std (0.45, 0.225)
      - BCTHW tensor layout (1, C, T, H, W)

    The model runs on CPU so it does not compete with Detic for GPU VRAM,
    keeping the T4's 15.6 GB entirely available for detection.
    Inference is dispatched to a background thread so it never blocks
    the main processing loop.
    """

    # Official Kinetics mean/std as specified in pytorchvideo documentation
    _MEAN      = [0.45, 0.45, 0.45]  # Per-channel mean used to normalise input frames (same for R, G, B)
    _STD       = [0.225, 0.225, 0.225]  # Per-channel standard deviation for normalisation (same for R, G, B)
    _SIDE_SIZE = 256   # Short-side resize target before centre cropping (official MViT preprocessing)
    _CROP_SIZE = 224   # Square spatial crop size applied after short-side scaling (official MViT preprocessing)

    def __init__(self, cfg: EyeConfig):
        self.cfg          = cfg  # Store EyeConfig reference for clip length, stride, and interval settings
        self.model        = None  # Will hold the loaded MViT model; None means action recognition is disabled
        self.labels       = _load_kinetics700_labels()  # Load Kinetics-700 class label strings at construction time
        # Buffer must hold clip_len * frame_stride raw frames so we can
        # sample every frame_stride-th frame and get exactly clip_len frames.
        _buf_size = cfg.action_clip_len * cfg.action_frame_stride  # Compute the total frame buffer size needed to fill one MViT clip
        self._clip_buffer: Dict[int, deque] = defaultdict(
            lambda: deque(maxlen=_buf_size))  # Per-track rolling frame buffer; old frames are automatically evicted
        self._last_action_frame: Dict[int, int] = {}  # Maps each track ID to the frame index when its last inference was triggered
        # Tracks which tids have had their very first inference triggered
        # so we fire immediately when the buffer first fills, not waiting
        # for the next action_interval_frames boundary.
        self._first_infer_done: set = set()  # Set of track IDs that have already completed their first inference
        self._lock        = threading.Lock()  # Thread lock protecting all internal buffers and state
        self._pool        = concurrent.futures.ThreadPoolExecutor(
            max_workers=1, thread_name_prefix="action_rec")  # Single-worker thread pool so inferences are serialised and never overload the CPU
        self._futures: Dict[int, concurrent.futures.Future] = {}  # Maps track IDs to their in-flight inference futures
        self._load()  # Load the MViT model weights during construction

    def _load(self):
        """
        Loads MViT-B 16x4 with official Kinetics-700 weights.

        Strategy:
          1. Load the model architecture via torch.hub WITHOUT pretrained weights
             (pretrained=True would load Kinetics-400, giving only 400 classes).
          2. Load the Kinetics-700 checkpoint downloaded during bootstrap.
          3. If the K700 checkpoint is missing, fall back to pretrained=True
             (K400, 400 classes) so the system still functions.
          4. Place the model on CPU to keep all GPU VRAM free for Detic.
        """
        _run_console.print("  [bold blue]Loading MViT-B 16x4 (Kinetics-700) from pytorchvideo hub...[/bold blue]")  # Inform the user that the MViT model is being loaded
        try:
            # Step 1: load architecture only (no pretrained weights yet)
            model = torch.hub.load(
                "facebookresearch/pytorchvideo:main",
                model="mvit_base_16x4",
                pretrained=False,   # Load architecture only; Kinetics-700 weights are applied in Step 2
            )  # Load the MViT-B 16x4 architecture from the pytorchvideo torch.hub entry point

            k700_path = self.cfg.mvit_k700_path  # Retrieve the configured path to the Kinetics-700 checkpoint
            if os.path.exists(k700_path) and os.path.getsize(k700_path) > 1_000_000:  # Verify the checkpoint file exists and is larger than 1 MB
                # Step 2: load Kinetics-700 checkpoint
                # The checkpoint stores the full model state dict under key "model_state"
                # as per the PySlowFast / pytorchvideo checkpoint convention.
                checkpoint = torch.load(k700_path, map_location="cpu")  # Load the checkpoint onto CPU to avoid GPU memory usage
                if "model_state" in checkpoint:
                    state_dict = checkpoint["model_state"]  # Extract state dict from PySlowFast-style checkpoint
                elif "state_dict" in checkpoint:
                    state_dict = checkpoint["state_dict"]  # Extract state dict from generic checkpoint format
                else:
                    state_dict = checkpoint  # Treat the entire checkpoint as the state dict if no known key is present

                # Strip any "model." prefix some checkpoints add
                cleaned = {
                    (k[len("model."):] if k.startswith("model.") else k): v
                    for k, v in state_dict.items()
                }  # Normalise key names by stripping a "model." prefix if present
                missing, unexpected = model.load_state_dict(cleaned, strict=False)  # Load the cleaned state dict, allowing missing or unexpected keys
                if missing:
                    log.debug(f"[MViT] Missing keys in state_dict: {len(missing)}")  # Log the count of missing keys at DEBUG level for diagnostics

                _run_console.print(
                    f"  [bold green]MViT-B 16x4 ready | K700 weights | "
                    f"{len(self.labels)} classes | device=CPU[/bold green]")  # Confirm successful load of the Kinetics-700 model
            else:
                # Fallback: use K400 pretrained weights (400 classes)
                _run_console.print(
                    "  [bold yellow]MViT K700 checkpoint not found; falling back to K400 pretrained.[/bold yellow]")  # Warn that K700 checkpoint is unavailable
                model = torch.hub.load(
                    "facebookresearch/pytorchvideo:main",
                    model="mvit_base_16x4",
                    pretrained=True,  # Load Kinetics-400 pretrained weights as a fallback
                )  # Load MViT-B 16x4 with the built-in K400 pretrained weights
                _run_console.print(
                    f"  [bold yellow]MViT-B 16x4 running with K400 weights (400 classes).[/bold yellow]")  # Inform user that the system is operating with fewer action classes

            # Place on CPU — keeps all GPU VRAM free for Detic's SwinB backbone
            self.model = model.eval().cpu()  # Switch the model to evaluation mode and move it to CPU

        except Exception as e:
            _run_console.print(
                f"  [bold yellow]MViT-B 16x4 load failed ({e}); action recognition disabled.[/bold yellow]")  # Warn that action recognition is unavailable due to a load error
            log.warning(f"[ActionRecognizer] load error: {e}")  # Log the error at WARNING level for the log file
            self.model = None  # Set model to None so all subsequent inference calls are skipped

    # ── frame buffer management ───────────────────────────────────────────────

    def push_frame(self, tid: int, crop_bgr: np.ndarray):
        """
        Appends a person crop (BGR, any size) to the temporal buffer for
        track `tid`.  Thread-safe.
        """
        with self._lock:
            self._clip_buffer[tid].append(crop_bgr)  # Thread-safely append the crop to the track's rolling frame buffer

    def _build_clip_tensor(self, tid: int) -> Optional[torch.Tensor]:
        """
        Assembles the temporal buffer into a (1, C, T, H, W) float32 tensor
        following the official PyTorchVideo MViT-B 16x4 preprocessing pipeline:
          1. Sample exactly action_clip_len frames with action_frame_stride spacing
          2. Resize shortest side to _SIDE_SIZE (256)
          3. Center-crop to (_CROP_SIZE, _CROP_SIZE) (224x224)
          4. Normalise with official Kinetics mean/std
          5. Stack along time axis → (C, T, H, W) then add batch dim

        The "16x4" in the model name means T=16 frames sampled every 4 frames.
        Sending any T ≠ 16 breaks the learned positional embeddings and causes
        the "size of tensor a must match size of tensor b" error.
        """
        with self._lock:
            frames = list(self._clip_buffer[tid])  # Snapshot the current frame buffer as a plain list

        needed = self.cfg.action_clip_len * self.cfg.action_frame_stride  # Total raw frames required to produce one full clip
        if len(frames) < needed:
            return None  # Not enough frames have accumulated yet; defer inference

        # Take the most recent `needed` frames and sample every frame_stride-th one.
        # This exactly replicates the temporal sampling used during training.
        recent  = frames[-needed:]  # Slice the most recent `needed` frames from the buffer
        sampled = [recent[i] for i in range(0, needed, self.cfg.action_frame_stride)]  # Sample every stride-th frame to produce exactly action_clip_len frames
        # Ensure we have exactly action_clip_len frames (guards rounding edge cases)
        sampled = sampled[:self.cfg.action_clip_len]  # Trim to exactly clip_len in case of off-by-one rounding

        processed = []  # List that will hold the preprocessed frame arrays
        mean = np.array(self._MEAN, dtype=np.float32) * 255  # Convert normalisation mean from [0,1] to [0,255] range
        std  = np.array(self._STD,  dtype=np.float32) * 255  # Convert normalisation std from [0,1] to [0,255] range

        for bgr in sampled:
            # Step 1 — ShortSideScale: resize so shortest side == _SIDE_SIZE
            h, w = bgr.shape[:2]  # Get current frame height and width
            if h < w:
                new_h, new_w = self._SIDE_SIZE, int(w * self._SIDE_SIZE / h)  # Height is the short side; scale height to target, preserve aspect ratio
            else:
                new_h, new_w = int(h * self._SIDE_SIZE / w), self._SIDE_SIZE  # Width is the short side; scale width to target, preserve aspect ratio
            resized = cv2.resize(bgr, (new_w, new_h), interpolation=cv2.INTER_LINEAR)  # Resize the frame to the computed dimensions using bilinear interpolation

            # Step 2 — CenterCrop to _CROP_SIZE x _CROP_SIZE
            ch, cw = resized.shape[:2]  # Get dimensions of the resized frame
            y0 = max((ch - self._CROP_SIZE) // 2, 0)  # Compute the top y coordinate for the centre crop
            x0 = max((cw - self._CROP_SIZE) // 2, 0)  # Compute the left x coordinate for the centre crop
            cropped = resized[y0:y0 + self._CROP_SIZE, x0:x0 + self._CROP_SIZE]  # Extract the centre crop region

            # Pad to exact crop size if the source was smaller than _CROP_SIZE
            if cropped.shape[0] < self._CROP_SIZE or cropped.shape[1] < self._CROP_SIZE:
                padded  = np.zeros((self._CROP_SIZE, self._CROP_SIZE, 3), dtype=np.uint8)  # Create a black canvas of the target crop size
                padded[:cropped.shape[0], :cropped.shape[1]] = cropped  # Copy the cropped content into the top-left of the canvas
                cropped = padded  # Use the padded version as the final crop

            # Step 3 — BGR→RGB and normalize
            rgb = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB).astype(np.float32)  # Convert BGR to RGB and cast to float32
            rgb = (rgb - mean) / (std + 1e-6)  # Subtract mean and divide by std to normalise; epsilon avoids division by zero
            processed.append(rgb)  # Append the normalised frame array to the processing list

        # Stack: list of (H,W,C) → (T,H,W,C) → (C,T,H,W) → (1,C,T,H,W)
        clip   = np.stack(processed, axis=0)                   # Stack frames along axis 0 to form (T, H, W, C)
        clip   = np.transpose(clip, (3, 0, 1, 2))              # Rearrange to (C, T, H, W) as required by PyTorch
        tensor = torch.from_numpy(clip).unsqueeze(0).float()   # Add batch dimension to get (1, C, T, H, W) and ensure float32

        return tensor  # Return the fully preprocessed clip tensor ready for MViT inference

    # ── inference ─────────────────────────────────────────────────────────────

    def _infer(self, tid: int) -> str:
        """Runs MViT-B 16x4 inference for track `tid` and returns the top action label."""
        if self.model is None:
            return "Unknown"  # Skip inference and return Unknown if the model failed to load
        try:
            clip = self._build_clip_tensor(tid)  # Build the preprocessed clip tensor for this track
            if clip is None:
                return "Unknown"  # Return Unknown if there are not enough frames to form a clip

            # MViT-B 16x4 expects a plain (1, C, T, H, W) tensor on CPU
            with torch.no_grad():
                logits = self.model(clip)  # Run forward pass through MViT; gradient computation is disabled

            # logits shape: (1, num_classes)
            probs   = torch.softmax(logits.float(), dim=-1)  # Convert raw logits to class probabilities via softmax
            top_idx = int(probs.argmax(dim=-1).item())  # Get the index of the highest-probability class
            action  = self.labels[top_idx] if top_idx < len(self.labels) else f"action_{top_idx}"  # Look up the class label, falling back to a numeric string if out of range

            # Capitalise and clean up label (underscores → spaces)
            action = action.replace("_", " ").title()  # Convert underscores to spaces and apply title-case formatting

            del clip, logits, probs  # Explicitly delete large tensors to free memory promptly
            gc.collect()  # Run garbage collection to reclaim freed tensor memory

            return action  # Return the cleaned top-1 action label

        except Exception as e:
            log.warning(f"[ActionRecognizer] tid={tid}: {e}")  # Log inference errors at WARNING level
            gc.collect()  # Attempt to recover memory even after an error
            return "Unknown"  # Return Unknown on any inference failure

    # ── async submit / retrieve ───────────────────────────────────────────────

    def submit(self, tid: int, frame_idx: int, store: "PersonAttributeStore"):
        """
        Non-blocking submit of an action inference task.

        Two trigger conditions (either fires an inference):
          1. The buffer just filled for the first time for this track — fires
             immediately so the first action label appears as early as possible
             (around frame 64 instead of waiting for the next interval boundary).
          2. At least action_interval_frames have passed since the last inference
             — the regular cadence for ongoing updates.
        """
        if self.model is None:
            return  # Skip submission entirely if the model is not loaded

        # Check buffer fullness for first-time trigger
        with self._lock:
            buf_len    = len(self._clip_buffer[tid])  # Read current buffer length for this track
            needed     = self.cfg.action_clip_len * self.cfg.action_frame_stride  # Total frames required for one clip
            first_full = (buf_len >= needed) and (tid not in self._first_infer_done)  # True if buffer just became full for the first time

        last         = self._last_action_frame.get(tid, -9999)  # Frame index of the last triggered inference (-9999 means never)
        interval_due = (frame_idx - last) >= self.cfg.action_interval_frames  # True if enough frames have elapsed for a regular inference

        if not (first_full or interval_due):
            return  # Neither trigger condition is met; skip this frame

        with self._lock:
            if tid in self._futures and not self._futures[tid].done():
                return  # An inference is already running for this track; do not submit a duplicate
            # Record the trigger so we do not re-fire the first-full condition
            self._first_infer_done.add(tid)  # Mark that first-full inference has been triggered for this track
            self._last_action_frame[tid] = frame_idx  # Update the last-triggered frame index

        def _task():
            result = self._infer(tid)  # Run the blocking MViT inference in the background thread
            store.put_action(tid, result)  # Store the result in the PersonAttributeStore

        with self._lock:
            self._futures[tid] = self._pool.submit(_task)  # Submit the inference task to the single-worker thread pool

    def purge(self, active_ids: set):
        """
        Clears frame buffers, futures and first-infer flags for track IDs
        that are no longer active.  Called periodically from process_frame.
        """
        with self._lock:
            stale = (set(self._clip_buffer.keys())
                     | set(self._futures.keys())
                     | self._first_infer_done) - active_ids  # Compute the set of track IDs that have data but are no longer active
            for tid in stale:
                self._clip_buffer.pop(tid, None)  # Discard the frame buffer for this stale track
                self._last_action_frame.pop(tid, None)  # Discard the last-inference-frame counter for this stale track
                self._first_infer_done.discard(tid)  # Clear the first-infer flag for this stale track
                fut = self._futures.pop(tid, None)  # Retrieve and discard the future for this stale track
                if fut and not fut.done():
                    fut.cancel()  # Attempt to cancel the in-flight future to avoid wasted work

    def flush_done(self):
        """Cleans up completed inference futures."""
        with self._lock:
            done = [k for k, f in self._futures.items() if f.done()]  # Collect keys of futures that have completed
            for k in done:
                try:    self._futures[k].result()  # Retrieve the result to propagate any exceptions
                except: pass  # Ignore exceptions from futures since results have already been stored
                self._futures.pop(k, None)  # Discard the completed future

    def shutdown(self):
        """Shuts down the action recognizer thread pool."""
        self._pool.shutdown(wait=False)  # Request immediate shutdown without blocking for pending tasks

_run_console.print("  [bold green]ActionRecognizer class defined.[/bold green]")  # Confirm that the ActionRecognizer class was defined successfully

# ============================================================ #
# ASYNC DEEPFACE POOL (emotion + age + gender)
# Manages asynchronous DeepFace analysis requests.
# ============================================================ #
class AsyncDeepFacePool:
    def __init__(self, cfg, store: PersonAttributeStore):
        self.cfg   = cfg  # Store EyeConfig reference for frame skip and backend settings
        self.store = store  # Store PersonAttributeStore reference for writing analysis results
        self._pool = concurrent.futures.ThreadPoolExecutor(
            max_workers=cfg.async_workers, thread_name_prefix="deepface")  # Thread pool sized to cfg.async_workers for concurrent DeepFace analysis
        self._futures: dict = {}  # Maps face analysis keys (e.g. "face_<tid>") to in-flight futures
        self._lock          = threading.Lock()  # Thread lock protecting the futures dictionary
        _run_console.print(
            f"  [bold green]AsyncDeepFacePool: {cfg.async_workers} workers (emotion+age+gender)[/bold green]")  # Confirm pool creation with the number of workers

    def _crop(self, frame, box):
        """Crops and pads a bounding box from the frame for face analysis."""
        x1, y1, x2, y2 = box  # Unpack the bounding box coordinates
        H, W            = frame.shape[:2]  # Get frame height and width for boundary clamping
        pw, ph          = int((x2-x1)*0.6), int((y2-y1)*0.5)  # Compute padding as a fraction of box width/height to include face context
        return frame[max(0,y1-ph):min(H,y2+ph), max(0,x1-pw):min(W,x2+pw)]  # Return the padded crop clamped to frame boundaries

    def submit_analysis(self, tid: int, frame: np.ndarray, box: List[int], frame_idx: int):
        """Submits a DeepFace analysis task if needed and not already running."""
        if frame_idx % self.cfg.deepface_frame_skip != 0:
            return  # Only analyse on every N-th frame to reduce CPU load

        need_emo    = self.store.emotion_needs_update(tid) and not self.store.is_analyzing_emotion(tid)  # True if emotion needs refreshing and no analysis is in progress
        need_age    = self.store.age_needs_update(tid)     and not self.store.is_analyzing_age(tid)  # True if age needs refreshing and no analysis is in progress
        need_gender = self.store.gender_needs_update(tid)  and not self.store.is_analyzing_gender(tid)  # True if gender needs refreshing and no analysis is in progress
        if not (need_emo or need_age or need_gender): return  # Nothing to analyse for this frame; skip

        actions = []  # List of DeepFace analysis actions to request
        if need_emo:    actions.append("emotion"); self.store.mark_analyzing_emotion(tid)  # Schedule emotion analysis and mark it as in-progress
        if need_age:    actions.append("age");     self.store.mark_analyzing_age(tid)  # Schedule age analysis and mark it as in-progress
        if need_gender: actions.append("gender");  self.store.mark_analyzing_gender(tid)  # Schedule gender analysis and mark it as in-progress

        key = f"face_{tid}"  # Unique key for this track's DeepFace future
        with self._lock:
            if key in self._futures and not self._futures[key].done():
                return  # A DeepFace task for this track is already in flight; do not submit a duplicate
        with self._lock:
            self._futures[key] = self._pool.submit(
                self._run_analysis, tid, frame.copy(), box, actions)  # Submit the DeepFace task; frame is copied to avoid concurrent modification

    def _run_analysis(self, tid: int, frame: np.ndarray, box: List[int], actions: List[str]):
        """Executes DeepFace analysis in a background thread."""
        from deepface import DeepFace  # Import DeepFace inside the thread to avoid module-level import issues
        crop    = self._crop(frame, box)  # Extract the padded face crop from the copied frame
        emotion = age = gender = "Unknown"  # Initialise all results to Unknown before analysis

        if crop.size > 0 and crop.shape[0] >= 32 and crop.shape[1] >= 32:  # Proceed only if the crop is large enough for reliable face analysis
            if crop.shape[0] < 100 or crop.shape[1] < 100:
                crop = cv2.resize(crop, (200, 200), interpolation=cv2.INTER_LINEAR)  # Upscale very small crops to 200×200 to improve DeepFace detection accuracy

            for backend in [self.cfg.deepface_backend, "opencv"]:  # Try the configured backend first, then fall back to the lightweight opencv backend
                try:
                    res = DeepFace.analyze(crop, actions=actions,
                                           enforce_detection=False,
                                           detector_backend=backend, silent=True)  # Run DeepFace analysis; enforce_detection=False avoids errors on ambiguous faces
                    if isinstance(res, list):
                        if not res: continue  # Skip if DeepFace returned an empty list (no faces detected)
                        res = res[0]  # Use the first detected face when multiple faces are present

                    if "emotion" in actions:
                        raw_emotion = str(res.get("dominant_emotion", "Unknown")).title()  # Extract the dominant emotion label and convert to title-case
                        emotion_scores = res.get("emotion", {})  # Get the full emotion confidence score dictionary
                        dominant_emotion_score = emotion_scores.get(raw_emotion.lower(), 0.0)  # Look up the confidence score for the dominant emotion
                        if dominant_emotion_score >= self.cfg.deepface_emotion_confidence_threshold:
                            emotion = raw_emotion  # Accept the emotion prediction if confidence meets the threshold
                        else:
                            emotion = "Unknown"  # Reject low-confidence emotion predictions
                    if "age" in actions:
                        raw_age = res.get("age", "Unknown")  # Extract the raw age estimate from DeepFace results
                        if isinstance(raw_age, (int, float)):
                            age_val = int(raw_age)  # Convert float age to integer for range bucketing
                            if age_val < 10: age = "0-10"  # Map age to the 0-10 decade bucket
                            elif age_val < 20: age = "10-20"  # Map age to the 10-20 decade bucket
                            elif age_val < 30: age = "20-30"  # Map age to the 20-30 decade bucket
                            elif age_val < 40: age = "30-40"  # Map age to the 30-40 decade bucket
                            elif age_val < 50: age = "40-50"  # Map age to the 40-50 decade bucket
                            elif age_val < 60: age = "50-60"  # Map age to the 50-60 decade bucket
                            elif age_val < 70: age = "60-70"  # Map age to the 60-70 decade bucket
                            elif age_val < 80: age = "70-80"  # Map age to the 70-80 decade bucket
                            else: age = "80+"  # Map any age 80 or above to the 80+ bucket
                        else:
                            age = str(raw_age)  # Keep non-numeric age values as-is
                    if "gender" in actions:
                        raw = str(res.get("dominant_gender", "Unknown")).lower()  # Extract and lowercase the dominant gender string
                        gender = "Man"   if raw in ("man",  "male")   else \
                                 "Woman" if raw in ("woman","female") else "Unknown"  # Normalise gender string to "Man", "Woman", or "Unknown"
                    log.info(f"[DeepFace] tid={tid} E={emotion} A={age} G={gender} [{backend}]")  # Log the successful analysis results for this track
                    break  # Exit backend loop after a successful analysis
                except ValueError:
                    log.debug(f"[DeepFace] tid={tid} [{backend}]: no face detected")  # Log at DEBUG level when the backend finds no face in the crop
                except Exception as e:
                    log.warning(f"[DeepFace] tid={tid} [{backend}]: {e}")  # Log unexpected errors at WARNING level

        if "emotion" in actions: self.store.put_emotion(tid, emotion)  # Write the emotion result (or Unknown) to the attribute store
        if "age"     in actions: self.store.put_age(tid, age)  # Write the age result (or Unknown) to the attribute store
        if "gender"  in actions: self.store.put_gender(tid, gender)  # Write the gender result (or Unknown) to the attribute store
        with self._lock:
            self._futures.pop(f"face_{tid}", None)  # Discard the completed future from the tracking dictionary to free memory

    def flush_done(self):
        """Processes completed DeepFace tasks and cleans up futures."""
        with self._lock:
            done = [k for k, f in self._futures.items() if f.done()]  # Collect keys of all finished futures
            for k in done:
                try:    self._futures[k].result()  # Retrieve result to surface any stored exceptions
                except: pass  # Ignore exceptions; results have already been stored in the PersonAttributeStore
                self._futures.pop(k, None)  # Discard the completed future

    def shutdown(self):
        """Shuts down the thread pool."""
        self._pool.shutdown(wait=False)  # Request pool shutdown without blocking for pending analysis tasks

# ============================================================ #
# ASYNC TRACKER BRIDGE
# Provides an asynchronous interface for the ByteTrack/IoU tracker.
# ============================================================ #
class AsyncTrackerBridge:
    # Number of frames to run synchronously at startup.
    # ByteTrack's Kalman filter needs a few frames of matched observations
    # before its predicted positions are reliable.  Running synchronously for
    # these warmup frames means the bounding boxes drawn on early frames
    # come from real detections, not uninitialized Kalman predictions.
    _SYNC_WARMUP_FRAMES = 8  # Number of initial frames processed synchronously to allow the Kalman filter to converge

    def __init__(self, tracker_factory):
        self._tracker    = tracker_factory()  # Instantiate the tracker using the provided factory callable
        self._lock       = threading.Lock()  # Thread lock protecting the cached latest tracking result
        self._latest     = []  # Cached tracking result returned to callers during async operation
        self._pool       = concurrent.futures.ThreadPoolExecutor(
            max_workers=1, thread_name_prefix="tracker")  # Single-worker pool for serialised async tracker updates
        self._future     = None  # Holds the in-flight tracking future; None when idle
        # Counter of synchronous calls remaining before switching to async mode
        self._sync_frames = self._SYNC_WARMUP_FRAMES  # Countdown to the end of the synchronous warmup phase
        _run_console.print("  [bold green]AsyncTrackerBridge online.[/bold green]")  # Confirm that the bridge has been initialised

    def update(self, detections, frame_shape):
        """
        Returns tracking results for the current frame.

        Warmup phase (first _SYNC_WARMUP_FRAMES calls): runs synchronously so
        ByteTrack's Kalman filter initialises with real observations.  This
        prevents the misaligned boxes seen in the first ~30 frames when the
        tracker returns Kalman-predicted positions before it has converged.

        Normal phase: async — submit current detections, return the result
        from the previous frame immediately so the main loop is never blocked.
        """
        if self._sync_frames > 0:
            # Synchronous warmup: block and return a real result every time
            self._sync_frames -= 1  # Decrement the warmup counter
            try:
                result = self._tracker.update(detections, frame_shape)  # Run the tracker synchronously to build up Kalman state
                with self._lock: self._latest = result  # Cache the synchronous result for future async returns
            except Exception as e:
                MONITOR.log_event(f"[AsyncTracker] warmup: {e}", "WARN")  # Log any warmup errors as warnings
            return list(self._latest)  # Return the freshly computed or cached result

        # Async phase: harvest previous result if ready
        if self._future is not None and self._future.done():
            try:
                with self._lock: self._latest = self._future.result()  # Store the completed tracking result
            except Exception as e:
                MONITOR.log_event(f"[AsyncTracker] {e}", "WARN")  # Log any async errors as warnings
            self._future = None  # Clear the completed future

        # Submit tracking for the next frame
        if self._future is None:
            self._future = self._pool.submit(self._tracker.update, detections, frame_shape)  # Submit the current frame's detections for async tracking

        with self._lock: return list(self._latest)  # Return the cached result from the previous frame without blocking

    def shutdown(self):
        """Shuts down the tracker thread pool."""
        self._pool.shutdown(wait=False)  # Request pool shutdown without waiting for in-flight tasks

# ============================================================ #
# DETECTOR
# Handles object detection using Detic or a COCO fallback.
# ============================================================ #
from detectron2.config         import get_cfg as d2_get_cfg  # Import detectron2 configuration builder under an alias
from detectron2.engine.defaults import DefaultPredictor  # Import the default detectron2 inference predictor
from detectron2.data           import MetadataCatalog  # Import the metadata catalog for registering class names

DETIC_AVAILABLE = False  # Flag indicating whether the Detic Python modules are importable
try:
    from detic.config          import add_detic_config  # Import the function that extends detectron2 config with Detic-specific fields
    from centernet.config      import add_centernet_config  # Import the function that extends the config with CenterNet2 fields
    from detic.modeling.utils  import reset_cls_test  # Import the utility that swaps in a new CLIP classifier at test time
    DETIC_AVAILABLE = True  # Mark Detic as importable
    _run_console.print("  [bold green]Detic modules imported.[/bold green]")  # Confirm successful Detic import
except ImportError:
    _run_console.print("[bold yellow]Detic not importable, falling back to COCO.[/bold yellow]")  # Warn that Detic is unavailable and the COCO fallback will be used

_BUILDIN_CLASSIFIER = {
    "lvis":       f"{_META_DIR}/lvis_v1_clip_a+cname.npy",  # Local path for the LVIS CLIP embedding file (redefined here for use by DeticDetector)
    "objects365": f"{_META_DIR}/o365_clip_a+cnamefix.npy",  # Local path for the Objects365 CLIP embedding file
    "openimages": f"{_META_DIR}/oid_clip_a+cname.npy",  # Local path for the OpenImages CLIP embedding file
    "coco":       f"{_META_DIR}/coco_clip_a+cname.npy",  # Local path for the COCO CLIP embedding file
}
_BUILDIN_METADATA_PATH = {
    "lvis": "lvis_v1_val", "objects365": "objects365_v2_val",  # Metadata catalog names for LVIS and Objects365 vocabularies
    "openimages": "oid_val_expanded", "coco": "coco_2017_val",  # Metadata catalog names for OpenImages and COCO vocabularies
}
_CLASSIFIER_URLS = {
    "lvis":       "https://dl.fbaipublicfiles.com/detic/lvis_v1_clip_a+cname.npy",  # Download URL for the LVIS CLIP classifier (redefined here for DeticDetector)
    "objects365": "https://dl.fbaipublicfiles.com/detic/o365_clip_a+cnamefix.npy",  # Download URL for the Objects365 CLIP classifier
    "openimages": "https://dl.fbaipublicfiles.com/detic/oid_clip_a+cname.npy",  # Download URL for the OpenImages CLIP classifier
    "coco":       "https://dl.fbaipublicfiles.com/detic/coco_clip_a+cname.npy",  # Download URL for the COCO CLIP classifier
}

def _ensure_classifier(vocab):
    """Ensures the specified vocabulary classifier is downloaded."""
    path = _BUILDIN_CLASSIFIER.get(vocab)  # Look up the local path for this vocabulary
    if path and not os.path.exists(path):  # Download only if the file is missing
        url = _CLASSIFIER_URLS.get(vocab)  # Look up the corresponding download URL
        if url:
            os.makedirs(os.path.dirname(path), exist_ok=True)  # Create the destination directory if needed
            try:   urllib.request.urlretrieve(url, path)  # Download the classifier file to the expected path
            except Exception as e:
                _run_console.print(f"[bold red]Classifier download failed for {vocab}: {e}[/bold red]")  # Report the download failure
                return None  # Return None to signal that the classifier is unavailable
    return path  # Return the local path to the classifier file

def _load_lvis_categories():
    """Loads LVIS categories, attempting from Detic module, local file, then download."""
    try:
        from detic.data.datasets.lvis_v1_categories import LVIS_CATEGORIES  # Try importing the auto-generated categories module from Detic
        return LVIS_CATEGORIES  # Return the imported list directly
    except ImportError:
        pass  # Fall through to the next loading strategy
    if os.path.exists(_CAT_FREQ_PATH):  # Check if the local category frequency JSON file is available
        try:
            with open(_CAT_FREQ_PATH) as f: cat_info = json.load(f)  # Load the JSON file from disk
            return [{"id": c.get("id", i+1), "name": c["name"]} for i, c in enumerate(cat_info)]  # Build a list of category dicts with id and name fields
        except:
            pass  # Fall through if the file is corrupt or unreadable
    try:
        with urllib.request.urlopen(_CAT_FREQ_PATH.replace(_META_DIR,
             "https://dl.fbaipublicfiles.com/detic"), timeout=20) as r:
            cat_info = json.loads(r.read())  # Download the category frequency JSON from the FAIR server
        categories = [{"id": c.get("id", i+1), "name": c["name"]} for i, c in enumerate(cat_info)]  # Build the category list from downloaded data
        out_path = "/content/Detic/detic/data/datasets/lvis_v1_categories.py"  # Path to write the regenerated categories module
        os.makedirs(os.path.dirname(out_path), exist_ok=True)  # Ensure the target directory exists
        with open(out_path, "w") as f:
            f.write("# Auto-generated LVIS categories\n")  # Write header comment to the module file
            f.write("LVIS_CATEGORIES = " + repr(categories) + "\n")  # Write the categories list as a Python literal
        return categories  # Return the downloaded and saved categories
    except:
        pass  # Fall through to the built-in minimal fallback list
    return [
        {"id":1,"name":"person"},   {"id":2,"name":"bicycle"},  {"id":3,"name":"car"},  # Minimal fallback: common person/vehicle categories
        {"id":62,"name":"chair"},   {"id":63,"name":"couch"},   {"id":65,"name":"bed"},  # Minimal fallback: common furniture categories
        {"id":67,"name":"dining table"}, {"id":72,"name":"tv"}, {"id":73,"name":"laptop"},  # Minimal fallback: common household object categories
        {"id":77,"name":"cell phone"},   {"id":84,"name":"book"},  # Minimal fallback: common personal item categories
    ]

class DeticDetector:
    def __init__(self, cfg):
        self.cfg         = cfg  # Store EyeConfig reference for threshold and path settings
        self.predictor   = None  # Will hold the detectron2 DefaultPredictor; None means detection is disabled
        self.class_names: List[str] = []  # List of class label strings indexed by class ID
        self._load()  # Load the detector at construction time

    def _load(self):
        """Initializes the Detic detector or a COCO fallback."""
        _run_console.print("  [bold blue]Loading DETIC detector...[/bold blue]")  # Inform the user that the detector is loading
        orig = os.getcwd()  # Remember the current working directory before changing it
        try:
            os.chdir(_DETIC_ROOT)  # Change to the Detic root so relative config paths resolve correctly
            self._load_detic()  # Attempt to load the Detic model
        except Exception as e:
            _run_console.print(f"[bold red]Detic load failed: {e}[/bold red]")  # Report the Detic load failure
            _run_console.print(traceback.format_exc())  # Print the full traceback for diagnosis
            self._load_fallback()  # Fall back to the COCO Mask R-CNN detector
        finally:
            os.chdir(orig)  # Always restore the original working directory

    def _load_detic(self):
        """Loads the Detic model configuration and weights."""
        if not DETIC_AVAILABLE: raise ImportError("Detic modules are not available for import.")  # Abort if Detic Python modules could not be imported
        vocab       = self.cfg.vocabulary  # Get the selected vocabulary name from configuration
        cfg_file    = f"{_DETIC_ROOT}/configs/Detic_LCOCOI21k_CLIP_SwinB_896b32_4x_ft4x_max-size.yaml"  # Path to the Detic Swin-B YAML config file
        wts_file    = f"{_MODELS_DIR}/Detic_LCOCOI21k_CLIP_SwinB_896b32_4x_ft4x_max-size.pth"  # Local path to the Detic Swin-B model weights
        wts         = wts_file if os.path.exists(wts_file) else \
                      "https://dl.fbaipublicfiles.com/detic/" \
                      "Detic_LCOCOI21k_CLIP_SwinB_896b32_4x_ft4x_max-size.pth"  # Use the local weights if present, otherwise provide the download URL as fallback

        d2cfg = d2_get_cfg()  # Create a fresh detectron2 configuration object
        add_centernet_config(d2cfg); add_detic_config(d2cfg)  # Extend the config with CenterNet2 and Detic-specific fields
        d2cfg.merge_from_file(cfg_file)  # Load the Detic Swin-B YAML into the config
        d2cfg.MODEL.WEIGHTS                           = wts  # Set the model weights path or URL
        d2cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST       = self.cfg.detic_threshold  # Apply the configured detection confidence threshold
        d2cfg.MODEL.ROI_BOX_HEAD.ZEROSHOT_WEIGHT_PATH = "rand"  # Use random zero-shot weights initially; they will be replaced by the CLIP classifier
        d2cfg.MODEL.ROI_HEADS.ONE_CLASS_PER_PROPOSAL  = True  # Assign only the top-scoring class to each proposal box
        d2cfg.MODEL.DEVICE                            = DEVICE  # Run the model on the selected compute device (cuda or cpu)
        d2cfg.MODEL.ROI_BOX_HEAD.CAT_FREQ_PATH        = _CAT_FREQ_PATH  # Provide the LVIS category frequency file for log-frequency adjustment

        if vocab in ("lvis", "objects365", "openimages", "coco"):
            categories = _load_lvis_categories()  # Load the category list for built-in vocabularies
            self.class_names = [c["name"].replace("_", " ") for c in categories]  # Build the class name list, converting underscores to spaces
        else:
            self.class_names = self._coco_classes()  # Use the standard COCO class names for unknown vocabularies

        meta_name = _BUILDIN_METADATA_PATH.get(vocab, "lvis_v1_val")  # Look up the catalog name for the chosen vocabulary
        if meta_name in MetadataCatalog.list(): MetadataCatalog.remove(meta_name)  # Clear any existing metadata entry to avoid stale class names
        MetadataCatalog.get(meta_name).set(thing_classes=self.class_names)  # Register the class names in the detectron2 metadata catalog
        d2cfg.MODEL.ROI_HEADS.NUM_CLASSES = len(self.class_names)  # Set the model's output class count to match the vocabulary size
        d2cfg.freeze()  # Lock the configuration to prevent accidental modification
        self.predictor = DefaultPredictor(d2cfg)  # Instantiate the detectron2 predictor with the configured model

        clf = _ensure_classifier(vocab)  # Ensure the CLIP classifier file is present locally
        if clf and os.path.exists(clf):
            zs = np.load(clf)  # Load the CLIP embedding matrix from the .npy file
            reset_cls_test(self.predictor.model, clf, zs.shape[0])  # Swap the model's classifier head with the loaded CLIP embeddings
        _run_console.print(
            f"  [bold green]DETIC detector ready | vocabulary={vocab} | {len(self.class_names)} classes.[/bold green]")  # Confirm successful Detic detector initialisation

    @staticmethod
    def _coco_classes():
        """Returns a list of standard COCO class names."""
        return [
            "person","bicycle","car","motorcycle","airplane","bus","train","truck","boat",  # COCO classes 1-9
            "traffic light","fire hydrant","stop sign","parking meter","bench","bird","cat","dog",  # COCO classes 10-17
            "horse","sheep","cow","elephant","bear","zebra","giraffe","backpack","umbrella",  # COCO classes 18-26
            "handbag","tie","suitcase","frisbee","skis","snowboard","sports ball","kite",  # COCO classes 27-34
            "baseball bat","baseball glove","skateboard","surfboard","tennis racket","bottle",  # COCO classes 35-40
            "wine glass","cup","fork","knife","spoon","bowl","banana","apple","sandwich","orange",  # COCO classes 41-50
            "broccoli","carrot","hot dog","pizza","donut","cake","chair","couch","potted plant",  # COCO classes 51-59
            "bed","dining table","toilet","tv","laptop","mouse","remote","keyboard","cell phone",  # COCO classes 60-68
            "microwave","oven","toaster","sink","refrigerator","book","clock","vase","scissors",  # COCO classes 69-77
            "teddy bear","hair drier","toothbrush",  # COCO classes 78-80
        ]

    def _load_fallback(self):
        """Loads a COCO-pretrained Mask R-CNN as a fallback detector."""
        _run_console.print("  [bold yellow]Activating COCO fallback detector...[/bold yellow]")  # Inform the user that the fallback detector is loading
        try:
            from detectron2 import model_zoo  # Import the detectron2 model zoo for pre-trained model references
            d2cfg = d2_get_cfg()  # Create a fresh detectron2 config
            d2cfg.merge_from_file(
                model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))  # Load the ResNet-50 FPN Mask R-CNN config from the model zoo
            d2cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(
                "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")  # Point to the pre-trained model zoo checkpoint URL
            d2cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = self.cfg.detic_threshold  # Apply the configured confidence threshold
            d2cfg.MODEL.DEVICE                      = DEVICE  # Run on the selected compute device
            self.class_names                        = self._coco_classes()  # Use the 80 standard COCO class names
            d2cfg.MODEL.ROI_HEADS.NUM_CLASSES       = len(self.class_names)  # Set output class count to 80
            d2cfg.freeze()  # Lock the fallback configuration
            self.predictor = DefaultPredictor(d2cfg)  # Instantiate the fallback predictor
            _run_console.print("  [bold green]Fallback COCO detector ready.[/bold green]")  # Confirm fallback detector is ready
        except Exception as e2:
            _run_console.print(f"  [bold red]All detectors failed to load: {e2}[/bold red]")  # Report that both Detic and the COCO fallback have failed

    def predict(self, frame, threshold_override=None):
        """
        Performs object detection on a given frame using float32 throughout.

        torch.cuda.amp.autocast() is intentionally NOT used here because
        detectron2's paste_masks_in_image is a CPU operation: when CUDA OOM
        forces tensors to CPU, FP16 (Half) tensors crash on
        grid_sampler_2d_cpu which has no Half kernel in PyTorch.
        Memory is managed instead by:
          - proactive cache clear before every inference pass
          - torch.inference_mode() to skip gradient bookkeeping
          - immediate del + cache clear after instances are extracted
          - a single OOM retry after a full cache flush
        """
        if self.predictor is None: return []  # Return empty list immediately if no predictor is loaded

        # Proactively clear fragmented GPU memory before inference
        if DEVICE == "cuda":
            torch.cuda.empty_cache()  # Release cached GPU memory before inference to reduce fragmentation

        def _run_inference():
            """Inner helper so we can retry once on OOM."""
            cur = self.predictor.cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST  # Read the current detection threshold from the config
            if threshold_override is not None and threshold_override != cur:
                self.predictor.cfg.defrost()  # Temporarily unfreeze the config to update the threshold
                self.predictor.cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = threshold_override  # Apply the override threshold
                self.predictor.cfg.freeze()  # Re-lock the config

            # inference_mode is lighter than no_grad and safe for detectron2
            with torch.inference_mode():
                out = self.predictor(frame)  # Run the detectron2 predictor on the input frame
            instances = out["instances"].to("cpu")  # Move detection results to CPU for NumPy extraction

            if threshold_override is not None and threshold_override != cur:
                self.predictor.cfg.defrost()  # Unfreeze config again to restore original threshold
                self.predictor.cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = cur  # Restore the original threshold value
                self.predictor.cfg.freeze()  # Re-lock the config

            results = []  # List to accumulate per-detection result dicts
            for i in range(len(instances)):
                cid   = int(instances.pred_classes[i])  # Get integer class ID for this detection
                label = self.class_names[cid] if cid < len(self.class_names) else f"class_{cid}"  # Map class ID to label name, using a numeric fallback if out of range
                results.append({
                    "box":    instances.pred_boxes[i].tensor.numpy()[0].astype(int).tolist(),  # Extract the bounding box as an [x1,y1,x2,y2] integer list
                    "label":  label,  # Class label string for this detection
                    "score":  float(instances.scores[i]),  # Detection confidence score as a Python float
                    "cls_id": cid,  # Integer class ID for this detection
                })
            del instances, out  # Explicitly delete detectron2 output tensors to free memory
            if DEVICE == "cuda":
                torch.cuda.empty_cache()  # Release GPU memory held by the deleted tensors
            return results  # Return the list of detection result dicts

        try:
            return _run_inference()  # Attempt a normal inference pass
        except torch.cuda.OutOfMemoryError:
            # First OOM: flush everything and retry once
            if DEVICE == "cuda":
                torch.cuda.empty_cache()  # Full GPU cache flush after the first OOM
            gc.collect()  # Run garbage collection to recover Python objects holding GPU tensors
            try:
                return _run_inference()  # Retry inference after the memory flush
            except torch.cuda.OutOfMemoryError:
                # Second consecutive OOM: skip this frame silently
                if DEVICE == "cuda":
                    torch.cuda.empty_cache()  # Best-effort cache flush after second OOM
                gc.collect()  # Run garbage collection again
                log.warning("[DeticDetector] CUDA OOM on retry — skipping frame.")  # Log the repeated OOM as a warning
                return []  # Return an empty detection list; the frame is skipped
        except Exception as e:
            _, success = self_heal("DeticDetector.predict", e)  # Attempt self-healing for non-OOM exceptions
            if not success:
                log.error("DeticDetector.predict failed after retries or with non-recoverable error.")  # Log that detection has permanently failed for this frame
            return []  # Return empty detections on unrecoverable error

_run_console.print("  [bold green]DeticDetector class defined.[/bold green]")  # Confirm that the DeticDetector class was defined successfully

# ============================================================ #
# TRACKER
# Uses ByteTrack for robust object tracking, with IoUTracker as a fallback.
# ============================================================ #
# Stub thop if missing (optional ByteTrack dependency to avoid import errors)
try:
    import thop  # noqa  # Try importing thop (model profiler used by ByteTrack's YOLOX)
except ModuleNotFoundError:
    import types as _t3  # Import types module to create a synthetic stub
    _thop         = _t3.ModuleType("thop")  # Create a minimal module object named "thop"
    _thop.profile = lambda model, inputs, **kw: (0, 0)  # Add a no-op profile function that returns zero FLOPs and zero params
    sys.modules["thop"] = _thop  # Register the stub so ByteTrack's import statement succeeds

BYTETRACK_OK = False  # Flag indicating whether ByteTrack imported without errors
try:
    from yolox.tracker.byte_tracker import BYTETracker  # Try importing the ByteTrack tracking class
    BYTETRACK_OK = True  # Mark ByteTrack as available
    _run_console.print("  [bold green]ByteTrack imported.[/bold green]")  # Confirm successful import
except Exception as _bt_err:
    _run_console.print(f"[bold yellow]ByteTrack unavailable ({_bt_err}); using IoU tracker.[/bold yellow]")  # Warn that the IoU fallback tracker will be used

# Simple arguments class for BYTETracker configuration
class SimpleArgs:
    track_thresh = CFG.track_thresh  # Minimum detection confidence to initialise a new track
    track_buffer = CFG.track_buffer  # Number of frames a lost track is kept before deletion
    match_thresh = CFG.match_thresh  # IoU threshold for matching detections to existing tracks
    mot20        = False  # Disable MOT20 dataset-specific ByteTrack behaviour

# Represents a single tracked object
class TrackedObject:
    _id_counter = 0  # Class-level counter for assigning unique track IDs
    def __init__(self, box, label, cls_id=None):
        TrackedObject._id_counter += 1  # Increment the shared counter to get a unique ID
        self.track_id = TrackedObject._id_counter  # Assign this object's unique track ID
        self.box = box; self.label = label; self.cls_id = cls_id  # Store the bounding box, class label, and class ID
        self.age = 0; self.lost = 0  # Initialise age (frames alive) and lost (frames without a match) counters
        self.history = deque(maxlen=30)  # Rolling history of the last 30 centre-point positions

    def update(self, box, label, cls_id=None):
        """Updates the object's state with new detection data."""
        self.box = box; self.label = label; self.cls_id = cls_id  # Store the updated bounding box and label
        self.age += 1; self.lost = 0  # Increment age and reset the lost counter since we have a new match
        self.history.append(((box[0]+box[2])//2, (box[1]+box[3])//2))  # Append the new centre point to the position history

def _iou(a, b):
    """Calculates Intersection over Union (IoU) between two bounding boxes."""
    ax1,ay1,ax2,ay2 = a; bx1,by1,bx2,by2 = b  # Unpack both bounding boxes
    ix,iy = max(ax1,bx1),max(ay1,by1); ix2,iy2 = min(ax2,bx2),min(ay2,by2)  # Compute intersection top-left and bottom-right corners
    inter = max(0,ix2-ix)*max(0,iy2-iy)  # Compute intersection area; clamp to 0 if boxes do not overlap
    union = (ax2-ax1)*(ay2-ay1)+(bx2-bx1)*(by2-by1)-inter  # Compute union area using inclusion-exclusion
    return inter/(union+1e-6)  # Return IoU, adding epsilon to avoid division by zero

class IoUTracker:
    """A basic IoU-based tracker as a fallback."""
    def __init__(self): self.tracks=[]; self.max_lost=15  # Initialise empty track list and set maximum lost-frame tolerance

    def update(self, detections, frame_shape=None):
        """Updates tracks based on new detections using IoU matching."""
        if not detections:
            for t in self.tracks: t.lost += 1  # Increment lost counter for every track when no detections arrive
            self.tracks = [t for t in self.tracks if t.lost < self.max_lost]  # Discard tracks that have been lost for too long
            return []  # Return an empty list since no active objects were detected

        matched_d, matched_t = set(), set()  # Sets tracking which detections and tracks have been matched
        for ti, trk in enumerate(self.tracks):
            best, bdi = 0.0, -1  # Initialise best IoU and best detection index for this track
            for di, d in enumerate(detections):
                if di in matched_d: continue  # Skip detections already matched to another track
                v = _iou(trk.box, d["box"])  # Compute IoU between the track box and this detection box
                if v > best: best, bdi = v, di  # Update best match if this IoU is higher
            if best > 0.3:
                trk.update(detections[bdi]["box"], detections[bdi]["label"], detections[bdi].get("cls_id"))  # Update the track with the best-matching detection
                matched_d.add(bdi); matched_t.add(ti)  # Record the matched detection and track indices

        for di, d in enumerate(detections):
            if di not in matched_d:
                self.tracks.append(TrackedObject(d["box"], d["label"], d.get("cls_id")))  # Create a new track for each unmatched detection

        for ti, trk in enumerate(self.tracks):
            if ti not in matched_t: trk.lost += 1  # Increment lost counter for tracks with no matching detection

        self.tracks = [t for t in self.tracks if t.lost < self.max_lost]  # Discard tracks that have exceeded the lost-frame limit
        return [{"track_id":t.track_id,"box":t.box,"label":t.label,"history":[]}
                for t in self.tracks if t.lost == 0]  # Return only currently matched tracks

class EyeTracker:
    """Selects and manages the primary object tracker (ByteTrack or IoU fallback)."""
    def __init__(self):
        if BYTETRACK_OK:
            try:
                SimpleArgs.track_thresh = CFG.track_thresh  # Apply configured track threshold to ByteTrack args
                SimpleArgs.track_buffer = CFG.track_buffer  # Apply configured track buffer to ByteTrack args
                SimpleArgs.match_thresh = CFG.match_thresh  # Apply configured match threshold to ByteTrack args
                self._bt   = BYTETracker(SimpleArgs(), frame_rate=CFG.max_fps)  # Instantiate ByteTracker with config-derived arguments
                self._mode = "bytetrack"  # Record that ByteTrack is active
            except Exception as e:
                _run_console.print(f"[bold yellow]ByteTrack initialization failed ({e}), falling back to IoU.[/bold yellow]")  # Warn that ByteTrack failed to initialise
                self._iou  = IoUTracker(); self._mode = "iou"  # Fall back to the IoU tracker
        else:
            self._iou = IoUTracker(); self._mode = "iou"  # Use IoU tracker since ByteTrack is not available
        _run_console.print(f"  [bold green]Tracker mode: {self._mode.upper()}[/bold green]")  # Print the active tracker mode

    def update(self, detections, frame_shape):
        """Delegates tracking update to the active tracker."""
        return self._update_bt(detections, frame_shape) if self._mode == "bytetrack" \
               else self._iou.update(detections, frame_shape)  # Route to ByteTrack or IoU tracker based on active mode

    def _update_bt(self, detections, frame_shape):
        """Handles updating ByteTrack with detections."""
        try:
            H, W = frame_shape[:2]  # Extract frame height and width for ByteTrack input
            if not detections:
                self._bt.update(np.empty((0,5),dtype=np.float32),[H,W],[H,W]); return []  # Feed an empty array to ByteTrack when there are no detections
            rows = [[float(d["box"][0]),float(d["box"][1]),float(d["box"][2]),float(d["box"][3]),
                     float(d.get("score",1.0))] for d in detections if "score" in d]  # Build a (N,5) float array of [x1,y1,x2,y2,score] rows for ByteTrack
            if not rows:
                self._bt.update(np.empty((0,5),dtype=np.float32),[H,W],[H,W]); return []  # Handle the case where all detections lack scores

            targets = self._bt.update(np.array(rows,dtype=np.float32),[H,W],[H,W])  # Run ByteTrack update and receive a list of active track objects
            results = []  # Accumulate formatted tracking results
            for t in targets:
                tlbr = t.tlbr.cpu().numpy() if isinstance(t.tlbr,torch.Tensor) else t.tlbr  # Convert the track's bounding box to a NumPy array if it is a tensor
                x1,y1,x2,y2 = map(int, tlbr)  # Extract integer pixel coordinates from the bounding box
                cx, cy = (x1+x2)/2.0, (y1+y2)/2.0  # Compute the centre of the ByteTrack bounding box
                relevant_detections = [
                    d for d in detections if
                    (d["box"][0] <= cx <= d["box"][2]) and (d["box"][1] <= cy <= d["box"][3])
                ]  # Find detections whose boxes contain the track centre point
                if not relevant_detections:
                    best_d = min(detections, key=lambda d: (
                        ((d["box"][0]+d["box"][2])/2-cx)**2+((d["box"][1]+d["box"][3])/2-cy)**2))  # Fall back to the spatially closest detection when no box contains the centre
                else:
                    best_d = min(relevant_detections, key=lambda d: (
                        ((d["box"][0]+d["box"][2])/2-cx)**2+((d["box"][1]+d["box"][3])/2-cy)**2))  # Choose the containing detection closest to the track centre

                results.append({"track_id":int(t.track_id),"box":[x1,y1,x2,y2],
                                "label":best_d.get("label","object"),
                                "cls_id":best_d.get("cls_id",-1),"history":[]})  # Append the formatted tracking result dict
            return results  # Return the list of active track results
        except Exception as e:
            MONITOR.log_event(f"ByteTrack error encountered: {e}", "WARN")  # Log the ByteTrack error and switch to the IoU fallback
            self._mode = "iou"; self._iou = IoUTracker()  # Switch permanently to the IoU tracker after a ByteTrack failure
            return self._iou.update(detections, frame_shape)  # Retry the update with the IoU tracker

_run_console.print("  [bold green]EyeTracker class defined.[/bold green]")  # Confirm that the EyeTracker class was defined successfully

# ============================================================ #
# FACE ANALYZER (warm-up only)
# Provides a mechanism to warm up the DeepFace library to avoid first-call delays.
# ============================================================ #
from deepface import DeepFace  # Import DeepFace at module level for the warm-up call

class FaceAnalyzer:
    def __init__(self, cfg):
        self.cfg = cfg  # Store EyeConfig reference for the backend setting
        self._warm_up()  # Immediately trigger the warm-up on construction

    def _warm_up(self):
        """Performs a dummy analysis to initialize DeepFace models."""
        _run_console.print("  [bold blue]Warming up DeepFace (emotion + age + gender)...[/bold blue]")  # Inform the user that DeepFace model loading is in progress
        try:
            DeepFace.analyze(np.zeros((160,160,3),dtype=np.uint8),
                             actions=["emotion","age","gender"],
                             enforce_detection=False,
                             detector_backend=self.cfg.deepface_backend,
                             silent=True)  # Run a dummy analysis on a black 160×160 image to pre-load all DeepFace models
            _run_console.print("  [bold green]DeepFace warm-up complete.[/bold green]")  # Confirm that warm-up succeeded
        except Exception as e:
            _run_console.print(f"[bold yellow]DeepFace warm-up skipped due to error: {e}[/bold yellow]")  # Warn that warm-up failed; DeepFace will initialise on first real use instead

_run_console.print("  [bold green]FaceAnalyzer class defined.[/bold green]")  # Confirm that the FaceAnalyzer class was defined successfully

# ============================================================ #
# DRAW ENGINE
# Renders multi-line label box above each tracked object.
# ============================================================ #
class DrawEngine:
    RED   = (0,  0, 220)  # BGR colour constant for red bounding box borders and accents
    WHITE = (230,230,230)  # BGR colour constant for label text
    CYAN  = (200,200,  0)  # BGR colour constant for HUD secondary information
    AMBER = (  0,165,255)  # BGR colour constant for HUD primary information

    def __init__(self, cfg):
        self.cfg    = cfg  # Store EyeConfig reference for font scale and box styling parameters
        self._font  = cv2.FONT_HERSHEY_SIMPLEX  # Standard sans-serif font used for label text
        self._fontm = cv2.FONT_HERSHEY_DUPLEX  # Slightly bolder font used for the HUD title

    def draw_l_box(self, frame, box, color=None, thickness=2, size=18):
        """Draws an 'L-shaped' border on the corners of a bounding box."""
        color      = color or self.RED  # Use provided colour or fall back to the default red
        x1,y1,x2,y2 = box; L, t = size, thickness  # Unpack box coordinates and set L-arm length and line thickness
        for p1, p2, p3 in [
            ((x1,y1),(x1+L,y1),(x1,y1+L)),  # Top-left corner: horizontal and vertical arms
            ((x2,y1),(x2-L,y1),(x2,y1+L)),  # Top-right corner: horizontal and vertical arms
            ((x1,y2),(x1+L,y2),(x1,y2-L)),  # Bottom-left corner: horizontal and vertical arms
            ((x2,y2),(x2-L,y2),(x2,y2-L)),  # Bottom-right corner: horizontal and vertical arms
        ]:
            cv2.line(frame,p1,p2,color,t,cv2.LINE_AA)  # Draw the horizontal arm of each L-corner
            cv2.line(frame,p1,p3,color,t,cv2.LINE_AA)  # Draw the vertical arm of each L-corner

        ov = frame.copy()  # Copy the frame to create a transparent overlay
        cv2.rectangle(ov,(x1,y1),(x2,y2),(0,0,40),-1)  # Fill the box interior with a dark blue colour on the overlay
        cv2.addWeighted(ov,0.06,frame,0.94,0,frame)  # Blend the overlay at 6% opacity for a subtle fill effect

    def draw_label(self, frame, box, lines: List[str], color=None):
        """Draws multi-line text labels above a bounding box."""
        color      = color or self.RED  # Use provided colour or fall back to default red
        x1,y1,x2,y2 = box  # Unpack bounding box coordinates
        lh, pad    = 16, 4  # Line height and vertical padding in pixels for the label background
        total_h    = lh * len(lines) + pad * 2  # Total height of the label background rectangle
        label_y    = max(y1 - total_h - 4, 0)  # Position the label above the box, clamping to the top of the frame

        max_w = max(
            cv2.getTextSize(ln, self._font, self.cfg.font_scale, 1)[0][0]
            for ln in lines
        )  # Compute the maximum text width across all label lines
        bx2 = min(x1 + max_w + 10, frame.shape[1])  # Compute the right edge of the label background, clamped to frame width

        ov = frame.copy()  # Copy the frame for the semi-transparent label background
        cv2.rectangle(ov,(x1,label_y),(bx2,label_y+total_h),(10,10,10),-1)  # Draw a dark background rectangle on the overlay
        cv2.addWeighted(ov,0.75,frame,0.25,0,frame)  # Blend the overlay at 75% opacity for a readable background
        cv2.rectangle(frame,(x1,label_y),(x1+3,label_y+total_h),color,-1)  # Draw a solid colour accent bar on the left edge of the label

        for i, ln in enumerate(lines):
            cv2.putText(frame, ln,
                        (x1+6, label_y+pad+lh*(i+1)),
                        self._font, self.cfg.font_scale, self.WHITE, 1, cv2.LINE_AA)  # Draw each label line in white with anti-aliasing

    def draw_hud(self, frame, monitor, n_objects, gov_level=0,
                 brightness=128.0, clahe_clip=2.0):
        """Draws a heads-up display with system metrics."""
        H, W = frame.shape[:2]  # Get frame dimensions for positioning HUD elements
        cv2.rectangle(frame,(0,0),(W,32),(5,5,5),-1)  # Draw a dark banner across the top of the frame for the HUD
        cv2.putText(frame," EyeOfAI  MULTI-MODAL INTELLIGENCE ENGINE",
                    (W//2-255,21),self._fontm,0.55,self.RED,1,cv2.LINE_AA)  # Draw the centred HUD title in red

        fps = monitor.avg_fps(); gpu = monitor.gpu_stats(); cpu = monitor.cpu_stats()  # Retrieve current FPS, GPU, and CPU metrics

        for i, txt in enumerate([f"FPS:{fps:.1f}",f"OBJ:{n_objects}",
                                  f"CPU:{cpu['pct']:.0f}%",f"RAM:{cpu['ram_gb']:.1f}G",
                                  f"BRIGHT:{brightness:.0f}",f"CLAHE:{clahe_clip:.1f}"]):
            cv2.putText(frame,txt,(10,50+i*15),self._font,0.4,self.AMBER,1,cv2.LINE_AA)  # Draw each left-side HUD metric in amber at 15-pixel vertical intervals

        for i, txt in enumerate([f"GPU:{gpu['used_gb']:.1f}/{gpu['total_gb']:.1f}G",
                                  f"UP:{monitor.uptime()}"]):
            cv2.putText(frame,txt,(W-160,50+i*15),self._font,0.4,self.CYAN,1,cv2.LINE_AA)  # Draw GPU memory and uptime on the right side in cyan

    def draw_scanline_effect(self, frame, intensity=0.04):
        """Applies a subtle scanline effect to the frame."""
        ov = np.zeros_like(frame); ov[::3,:] = 0  # Create a zero overlay and zero out every third row (scanlines are transparent)
        cv2.addWeighted(ov,intensity,frame,1.0-intensity,0,frame)  # Blend the scanline overlay at low intensity for a subtle CRT effect

DRAWER = DrawEngine(CFG)  # Instantiate the global DrawEngine with the default configuration
_run_console.print("  [bold green]DrawEngine ready.[/bold green]")  # Confirm that the draw engine has been initialised

# ============================================================ #
# MASTER PIPELINE
# Integrates all components for real-time multi-modal analysis.
# ============================================================ #
class EyeOfAIPipeline:
    def __init__(self, cfg=CFG):
        self.cfg    = cfg  # Store the EyeConfig instance used for all pipeline parameters
        self.monitor = MONITOR  # Reference the global SystemMonitor for metrics and event logging
        self.drawer = DRAWER  # Reference the global DrawEngine for frame annotation
        _run_console.print("  [bold blue]Initializing EyeOfAI pipeline...[/bold blue]")  # Announce that pipeline initialisation is starting

        orig = os.getcwd()  # Save the current working directory before switching to Detic root
        try:
            os.chdir(_DETIC_ROOT)  # Switch to Detic root so relative config paths inside DeticDetector resolve correctly
            self.detector = DeticDetector(cfg)  # Instantiate and load the Detic (or fallback) detector
        finally:
            os.chdir(orig)  # Restore the original working directory after detector loading

        self.governor = PerformanceGovernor(cfg, MONITOR)  # Create the Performance Governor for dynamic quality control
        MONITOR.governor = self.governor  # Register the governor with the monitor so PersonAttributeStore can access it

        self._tracker       = EyeTracker()  # Instantiate the object tracker (ByteTrack or IoU fallback)
        self._face_svc      = FaceAnalyzer(cfg)  # Instantiate and warm up the DeepFace analyser
        self.calibrator     = SceneCalibrator(cfg)  # Instantiate the adaptive CLAHE scene calibrator
        self._person_store  = PersonAttributeStore(cfg)  # Instantiate the per-track attribute store
        self._async_face    = AsyncDeepFacePool(cfg, self._person_store)  # Instantiate the async DeepFace worker pool
        self._async_tracker = AsyncTrackerBridge(lambda: EyeTracker())  # Instantiate the async tracker bridge with a tracker factory
        self._action_rec    = ActionRecognizer(cfg)  # Instantiate and load the MViT action recogniser
        self._stop_evt      = threading.Event()  # Event used to signal a graceful pipeline shutdown
        self._frame_cnt     = 0  # Counter tracking the total number of frames processed by this pipeline

        MONITOR.log_event("Pipeline initialized", "INFO")  # Log a pipeline initialisation event
        _run_console.print("  [bold green]EyeOfAI Pipeline ready.[/bold green]")  # Confirm the pipeline is fully initialised

    @staticmethod
    def is_person(label):
        """Checks if a given label refers to a person."""
        return any(kw in label.lower() for kw in
                   ["person","human","man","woman","people","pedestrian","face","body","kid","child"])  # Return True if any person-related keyword appears in the lowercased label

    def _resize(self, frame: np.ndarray, w: int, h: int) -> np.ndarray:
        """Resizes frame if dimensions do not match target."""
        fh, fw = frame.shape[:2]  # Get current frame height and width
        interp = cv2.INTER_AREA if w < fw else cv2.INTER_LINEAR  # Use INTER_AREA for downscaling and INTER_LINEAR for upscaling
        return cv2.resize(frame, (w, h), interpolation=interp) if (fw != w or fh != h) else frame  # Resize only if dimensions differ from target

    def process_frame(self, frame):
        """Processes a single video frame with detection, tracking, and analysis."""
        t0 = time.time()  # Record the start time for this frame's processing

        # Determine governor-adjusted resolution for detection
        ew, eh = self.governor.effective_resolution(self.cfg.input_width, self.cfg.input_height)  # Get the effective detection resolution from the governor

        # Build the detection frame at (possibly reduced) governor resolution
        frame_det = self._resize(frame, ew, eh)  # Resize the input frame to the governor-adjusted detection resolution
        frame_det = self.calibrator.calibrate(frame_det)  # Apply adaptive CLAHE brightness calibration to the detection frame

        thresh = max(0.10, min(0.90,
            self.governor.effective_threshold(self.cfg.detic_threshold) + self.calibrator.conf_offset))  # Compute the effective detection threshold combining governor adjustment and brightness offset

        # Run detection on the (possibly reduced) frame
        try:    detections = self.detector.predict(frame_det, threshold_override=thresh)  # Run object detection on the calibrated detection frame
        except Exception as e: self_heal("object_detection",e); detections=[]  # Self-heal on detection errors and continue with empty detections

        # Update object tracks asynchronously (tracks are in detection-frame coordinates)
        try:    tracks = self._async_tracker.update(detections, frame_det.shape)  # Submit detections to the async tracker and retrieve the previous frame's results
        except Exception as e: self_heal("object_tracking",e);  tracks=[]  # Self-heal on tracking errors and continue with empty tracks

        # Build the output frame at the configured output resolution
        frame_out = self._resize(frame, self.cfg.input_width, self.cfg.input_height)  # Resize the original frame to the full configured output resolution

        # Scale bounding box coordinates from detection resolution to output resolution
        if ew != self.cfg.input_width or eh != self.cfg.input_height:  # Only scale if detection and output resolutions differ
            sx = self.cfg.input_width  / ew  # Horizontal scale factor from detection to output resolution
            sy = self.cfg.input_height / eh  # Vertical scale factor from detection to output resolution
            scaled_tracks = []  # Accumulate rescaled track dicts
            for trk in tracks:
                x1, y1, x2, y2 = trk["box"]  # Unpack the box in detection-resolution coordinates
                scaled_tracks.append(dict(trk, box=[
                    int(x1 * sx), int(y1 * sy),
                    int(x2 * sx), int(y2 * sy),
                ]))  # Build a new track dict with the box scaled to output resolution
            tracks = scaled_tracks  # Replace the original tracks with the rescaled versions

        self.monitor.push_detection(len(tracks))  # Record the number of active tracks for this frame
        active_ids = {t["track_id"] for t in tracks}  # Build the set of currently active track IDs

        if self._frame_cnt % 60 == 0:  # Perform housekeeping every 60 frames to avoid per-frame overhead
            # Clear attributes for tracks that have disappeared
            self._person_store.purge_stale(active_ids)  # Clear attribute data for track IDs no longer in the active set
            self._async_face.flush_done()  # Process and discard completed DeepFace futures
            # purge() clears buffers AND _first_infer_done for gone tracks;
            # flush_done() is called inside purge so we only need purge here.
            self._action_rec.purge(active_ids)  # Clear action recognition buffers and futures for gone tracks

        for trk in tracks:
            box   = trk["box"]; label = trk["label"]; tid = trk["track_id"]  # Extract box, label, and track ID for this track
            x1,y1,x2,y2 = box  # Unpack the scaled bounding box coordinates

            # Crop from the output-resolution frame for face/action analysis
            crop  = frame_out[max(0,y1):min(frame_out.shape[0],y2),
                              max(0,x1):min(frame_out.shape[1],x2)]  # Extract the bounding box crop from the full-resolution output frame

            if self.is_person(label):
                if crop.size > 0 and crop.shape[0] >= 32 and crop.shape[1] >= 32:  # Only submit analysis for crops large enough to contain a face or body
                    # Submit DeepFace (emotion/age/gender)
                    self._async_face.submit_analysis(tid, frame_out, box, self._frame_cnt)  # Submit async DeepFace analysis for this person crop
                    # Push frame to action recognizer clip buffer
                    self._action_rec.push_frame(tid, crop)  # Append the crop to the action recogniser's temporal buffer
                    # Submit action inference
                    self._action_rec.submit(tid, self._frame_cnt, self._person_store)  # Conditionally trigger async MViT inference

                a = self._person_store.get(tid)  # Retrieve the latest emotion, age, gender, and action attributes for this track
                lines = [
                    f"Person (Gender:{a['gender']}, Age:{a['age']}, Emotion:{a['emotion']}, Action:{a['action']})"
                ]  # Build a single-line label combining all four person attributes
            else:
                lines = [label.title()]  # For non-person objects, use only the title-cased class label

            # Draw on the correctly sized output frame
            self.drawer.draw_l_box(frame_out, box, color=(0,0,220),
                                   thickness=self.cfg.box_thickness, size=self.cfg.l_size)  # Draw the L-shaped corner markers around the bounding box
            self.drawer.draw_label(frame_out, box, lines)  # Draw the text label above the bounding box

        if self.cfg.show_hud:
            self.drawer.draw_hud(frame_out, self.monitor, len(tracks), gov_level=self.governor.level,
                                 brightness=self.calibrator.brightness, clahe_clip=self.calibrator.clahe_clip)  # Overlay the performance HUD if enabled
            self.drawer.draw_scanline_effect(frame_out)  # Apply the scanline effect when the HUD is active

        elapsed = time.time() - t0  # Compute total processing time for this frame
        fps     = 1.0 / max(elapsed, 1e-6)  # Compute instantaneous FPS, guarding against division by zero
        self.monitor.push_fps(fps)  # Record this frame's FPS in the rolling history
        self.governor.tick(fps)  # Update the Performance Governor with the latest FPS measurement
        self._frame_cnt += 1  # Advance the frame counter
        self.monitor.frame_count += 1  # Advance the monitor's cumulative frame count

        rem = self.governor.target_frame_duration() - elapsed  # Compute remaining time budget for this frame
        if rem > 0.002: time.sleep(rem)  # Sleep to pace the pipeline to the target FPS if time budget remains
        return frame_out  # Return the annotated output frame

    # ── Pipeline Run Modes ───────────────────────────────────────────────────

    def run_colab(self, source=0, max_frames=300, output_path="/content/eyeofai_output.mp4"):
        """Runs the pipeline in a Google Colab environment."""
        from IPython.display import display as ipy_display, Image as IpyImage  # Import Colab display utilities
        cap = self._open_source(source)  # Open the video capture source
        if not cap:
            _run_console.print("[bold red]Cannot open video source.[/bold red]"); return  # Abort if the source could not be opened
        writer = self._make_writer(cap, output_path)  # Create the video writer for saving output
        _run_console.print(f"  [bold blue]Colab run initiated: {max_frames} frames → {output_path}[/bold blue]")  # Log the start of the Colab run
        frame_i = 0  # Local frame counter for this run session
        try:
            while frame_i < max_frames and not self._stop_evt.is_set():  # Loop until max_frames reached or stop is signalled
                ok, frame = cap.read()  # Read the next frame from the capture source
                if not ok: break  # End the loop if the source is exhausted
                out = self.process_frame(frame)  # Process the frame through the full pipeline
                if writer: writer.write(out)  # Write the annotated frame to the output video

                if frame_i % 30 == 0:  # Display a preview and status update every 30 frames
                    _, buf = cv2.imencode(".jpg", out, [cv2.IMWRITE_JPEG_QUALITY, 85])  # Encode the frame as JPEG at 85% quality
                    ipy_display(IpyImage(data=buf.tobytes()))  # Display the JPEG preview inline in the Colab notebook
                    _run_console.print(
                        f"  [bold green]Frame {frame_i}/{max_frames}[/bold green]"
                        f" | FPS:[cyan]{self.monitor.avg_fps():.1f}[/cyan]"
                        f" | GOV:[magenta]{self.governor.level_name}[/magenta]"
                        f" | BRIGHT:[yellow]{self.calibrator.brightness:.0f}[/yellow]")  # Print a status update with FPS, governor level, and brightness
                frame_i += 1  # Advance the frame counter
        except KeyboardInterrupt:
            _run_console.print("  [bold yellow]Processing interrupted by user.[/bold yellow]")  # Handle user interruption gracefully
        finally:
            cap.release()  # Release the video capture resource
            if writer: writer.release()  # Release the video writer resource
            self._async_face.shutdown()  # Shut down the DeepFace worker pool
            self._async_tracker.shutdown()  # Shut down the async tracker bridge
            self._action_rec.shutdown()  # Shut down the action recogniser thread pool
            _run_console.print(f"  [bold green]Output video saved: {output_path}[/bold green]")  # Confirm that the output video was saved

    def run_webcam(self, source=0, display=True, output_path=None):
        """Runs the pipeline with a webcam source and optional display/recording."""
        cap    = self._open_source(source)  # Open the webcam capture source
        writer = self._make_writer(cap, output_path) if (cap and output_path) else None  # Create video writer only if a path is provided
        if not cap: return  # Abort if the source could not be opened
        try:
            while not self._stop_evt.is_set():  # Loop until a stop is signalled
                ok, frame = cap.read()  # Read the next webcam frame
                if not ok: break  # End the loop if the webcam stream is lost
                out = self.process_frame(frame)  # Process the frame through the full pipeline
                if writer: writer.write(out)  # Write the annotated frame if recording is enabled
                if display:
                    cv2.imshow(" EyeOfAI ", out)  # Display the annotated frame in an OpenCV window
                    if cv2.waitKey(1) & 0xFF in (ord("q"), 27): break  # Exit on 'q' or Escape key press
        except KeyboardInterrupt:
            _run_console.print("  [bold yellow]Webcam processing interrupted.[/bold yellow]")  # Handle user interruption gracefully
        finally:
            cap.release()  # Release the webcam capture resource
            if writer: writer.release()  # Release the video writer if recording was active
            cv2.destroyAllWindows()  # Close all OpenCV display windows
            self._async_face.shutdown()  # Shut down the DeepFace worker pool
            self._async_tracker.shutdown()  # Shut down the async tracker bridge
            self._action_rec.shutdown()  # Shut down the action recogniser thread pool

    def _open_source(self, source):
        """Opens a video capture source, retrying if necessary."""
        for _ in range(3):  # Attempt up to 3 times to open the source
            cap = cv2.VideoCapture(source)  # Try to open the video source (file path or device index)
            if cap.isOpened():
                try:
                    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  self.cfg.input_width)  # Request the configured capture width
                    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, self.cfg.input_height)  # Request the configured capture height
                    cap.set(cv2.CAP_PROP_FPS,          self.cfg.max_fps)  # Request the configured maximum frame rate
                    cap.set(cv2.CAP_PROP_BUFFERSIZE,   2)  # Limit the internal capture buffer to 2 frames to reduce latency
                except: pass  # Ignore failures to set capture properties (e.g. read-only video files)
                return cap  # Return the successfully opened capture object
            time.sleep(0.5)  # Wait briefly before retrying if the source failed to open
        _run_console.print(f"[bold red]Cannot open video source: {source}[/bold red]")  # Report that all attempts to open the source failed
        return None  # Return None to signal that the source is unavailable

    def _make_writer(self, cap, path):
        """Creates a VideoWriter object for saving output video."""
        try:
            fps = cap.get(cv2.CAP_PROP_FPS) or self.cfg.max_fps  # Read FPS from the capture source, falling back to config value
            return cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*"mp4v"),
                                   fps, (self.cfg.input_width, self.cfg.input_height))  # Create an MP4V-encoded VideoWriter at the configured output resolution
        except Exception as e:
            _run_console.print(f"[bold yellow]VideoWriter initialization failed: {e}[/bold yellow]")  # Warn that video writing will be unavailable
            return None  # Return None so callers skip writing

    def stop(self):
        """Signals the pipeline to stop gracefully."""
        self._stop_evt.set()  # Set the stop event to signal all run loops to terminate

    def status(self):
        """Prints a detailed status report of the pipeline."""
        gpu = self.monitor.gpu_stats(); cpu = self.monitor.cpu_stats()  # Retrieve current GPU and CPU statistics
        _run_console.print(Panel(
            f"  Uptime      : {self.monitor.uptime()}\n"
            f"  Avg FPS     : {self.monitor.avg_fps():.1f}\n"
            f"  Objects     : {self.monitor.total_detections}\n"
            f"  Frames      : {self.monitor.frame_count}\n"
            f"  GOV Level   : {self.governor.level_name}\n"
            f"  Brightness  : {self.calibrator.brightness:.1f}\n"
            f"  CLAHE Clip  : {self.calibrator.clahe_clip:.2f}\n"
            f"  GPU         : {gpu['used_gb']:.1f}/{gpu['total_gb']:.1f} GB ({gpu['pct']}%)\n"
            f"  CPU         : {cpu['pct']}%  RAM:{cpu['ram_gb']:.1f}/{cpu['ram_total_gb']:.1f}GB\n"
            f"  Detector    : {'DETIC 21k' if self.detector.predictor else 'COCO FALLBACK'}\n"
            f"  Action Model: {'MViT-B 16x4 / Kinetics-700' if self._action_rec.model else 'DISABLED'}\n"
            f"  Errors      : {len(self.monitor.error_log)}",
            title="[bold red] EyeOfAI - SYSTEM STATUS [/bold red]",
            border_style="red"))  # Print a rich Panel containing all pipeline status metrics

_run_console.print("  [bold green]EyeOfAIPipeline class defined.[/bold green]")  # Confirm that the pipeline class was defined successfully

# ============================================================ #
# CONFIGURATION DASHBOARD (ipywidgets)
# Provides an interactive UI for adjusting pipeline parameters in Colab.
# ============================================================ #
try:
    import ipywidgets as widgets  # Import ipywidgets for interactive Colab UI elements
    from IPython.display import display as ipy_display, clear_output  # Import display utilities for rendering widgets
    WIDGETS_OK = True  # Mark ipywidgets as available
except ImportError:
    WIDGETS_OK = False  # Mark ipywidgets as unavailable
    _run_console.print("[bold yellow]ipywidgets not available. Configuration dashboard disabled.[/bold yellow]")  # Warn that the dashboard cannot be shown

_active_pipeline: Optional[EyeOfAIPipeline] = None  # Module-level reference to the active pipeline for live config updates

def launch_config_dashboard(pipeline=None):
    """Launches an interactive configuration dashboard if running in an IPython environment."""
    global _active_pipeline  # Declare intent to update the module-level active pipeline reference
    if pipeline is not None: _active_pipeline = pipeline  # Store the provided pipeline for live updates
    if not WIDGETS_OK:
        _run_console.print("[bold yellow]ipywidgets not available for dashboard.[/bold yellow]"); return  # Exit early if widgets are unavailable

    style  = {"description_width": "210px"}  # Widget style dict that allocates 210px for label descriptions
    layout = widgets.Layout(width="540px")  # Standard widget layout with 540px width

    thresh_w          = widgets.FloatSlider(value=CFG.detic_threshold,        min=0.10,max=0.90,step=0.05, description="Detection Threshold",      style=style,layout=layout,readout_format=".2f")  # Slider for adjusting the detection confidence threshold
    track_w           = widgets.FloatSlider(value=CFG.track_thresh,            min=0.10,max=0.90,step=0.05, description="Track Threshold",           style=style,layout=layout,readout_format=".2f")  # Slider for adjusting the ByteTrack association threshold
    fps_w             = widgets.IntSlider(  value=CFG.max_fps,                 min=5,   max=60,  step=5,    description="Max FPS",                   style=style,layout=layout)  # Slider for adjusting the maximum frames per second
    gov_fps_w         = widgets.FloatSlider(value=CFG.gov_target_fps,          min=10,  max=60,  step=1.0,  description="Gov Target FPS",            style=style,layout=layout,readout_format=".0f")  # Slider for adjusting the Performance Governor's target FPS
    emotion_w         = widgets.FloatSlider(value=CFG.emotion_interval_sec,    min=0.5, max=30,  step=0.5,  description="Emotion Interval (sec)",    style=style,layout=layout,readout_format=".1f")  # Slider for the minimum seconds between emotion analyses
    age_gender_w      = widgets.FloatSlider(value=CFG.age_gender_interval_sec, min=1.0, max=60,  step=1.0,  description="Age/Gender Interval (sec)", style=style,layout=layout,readout_format=".1f")  # Slider for the minimum seconds between age/gender analyses
    df_skip_w         = widgets.IntSlider(  value=CFG.deepface_frame_skip,     min=1,   max=30,  step=1,    description="DeepFace Frame Skip",       style=style,layout=layout)  # Slider for the DeepFace frame-skip interval
    df_emo_smooth_w   = widgets.IntSlider(  value=CFG.deepface_emotion_smooth_window, min=1, max=10, step=1, description="DF Emotion Smooth Window",  style=style,layout=layout)  # Slider for the emotion smoothing window size
    df_emo_conf_w     = widgets.FloatSlider(value=CFG.deepface_emotion_confidence_threshold, min=0.0, max=1.0, step=0.05, description="DF Emotion Min Conf", style=style,layout=layout,readout_format=".2f")  # Slider for the minimum emotion confidence threshold
    workers_w         = widgets.IntSlider(  value=CFG.async_workers,           min=1,   max=4,              description="Async Workers",             style=style,layout=layout)  # Slider for the number of async DeepFace worker threads
    show_hud_w        = widgets.Checkbox(   value=CFG.show_hud,                                             description="Show HUD overlay",          style=style,layout=layout)  # Checkbox for toggling the HUD overlay
    action_interval_w = widgets.IntSlider(  value=CFG.action_interval_frames,  min=4,   max=64,  step=4,    description="Action Interval (frames)",  style=style,layout=layout)  # Slider for the action recognition inference interval in frames

    apply_btn         = widgets.Button(description="  Apply Config", button_style="danger",
                                       layout=widgets.Layout(width="220px", height="38px"))  # Red "Apply Config" button to commit widget values to the pipeline
    status_out = widgets.Output()  # Output widget that captures and displays console messages after apply

    def _sync():
        """Synchronizes widget values with the global CFG object and the active pipeline."""
        CFG.detic_threshold        = thresh_w.value  # Write detection threshold from slider to global config
        CFG.track_thresh           = track_w.value  # Write track threshold from slider to global config
        CFG.max_fps                = fps_w.value  # Write max FPS from slider to global config
        CFG.gov_target_fps         = gov_fps_w.value  # Write governor target FPS from slider to global config
        CFG.emotion_interval_sec   = emotion_w.value  # Write emotion interval from slider to global config
        CFG.age_gender_interval_sec= age_gender_w.value  # Write age/gender interval from slider to global config
        CFG.deepface_frame_skip    = df_skip_w.value  # Write DeepFace frame skip from slider to global config
        CFG.deepface_emotion_smooth_window = df_emo_smooth_w.value  # Write emotion smoothing window from slider to global config
        CFG.deepface_emotion_confidence_threshold = df_emo_conf_w.value  # Write emotion confidence threshold from slider to global config
        CFG.async_workers          = workers_w.value  # Write async workers from slider to global config
        CFG.show_hud               = show_hud_w.value  # Write HUD toggle from checkbox to global config
        CFG.action_interval_frames = action_interval_w.value  # Write action interval from slider to global config

        if _active_pipeline is not None:
            p = _active_pipeline  # Alias the active pipeline for cleaner property assignments
            p.cfg.detic_threshold        = CFG.detic_threshold  # Apply detection threshold to the running pipeline
            p.cfg.track_thresh           = CFG.track_thresh  # Apply track threshold to the running pipeline
            p.cfg.max_fps                = CFG.max_fps  # Apply max FPS to the running pipeline
            p.cfg.gov_target_fps         = CFG.gov_target_fps  # Apply governor target FPS to the running pipeline
            p.cfg.emotion_interval_sec   = CFG.emotion_interval_sec  # Apply emotion interval to the running pipeline
            p.cfg.age_gender_interval_sec= CFG.age_gender_interval_sec  # Apply age/gender interval to the running pipeline
            p.cfg.deepface_frame_skip    = CFG.deepface_frame_skip  # Apply DeepFace frame skip to the running pipeline
            p.cfg.deepface_emotion_smooth_window = CFG.deepface_emotion_smooth_window  # Apply smoothing window to the running pipeline
            p.cfg.deepface_emotion_confidence_threshold = CFG.deepface_emotion_confidence_threshold  # Apply confidence threshold to the running pipeline
            p.cfg.async_workers          = CFG.async_workers  # Apply async workers setting to the running pipeline
            p.cfg.show_hud               = CFG.show_hud  # Apply HUD toggle to the running pipeline
            p.cfg.action_interval_frames = CFG.action_interval_frames  # Apply action interval to the running pipeline

            p.governor.cfg.gov_target_fps               = CFG.gov_target_fps  # Propagate governor target FPS to the governor object
            p._person_store.cfg.emotion_interval_sec    = CFG.emotion_interval_sec  # Propagate emotion interval to the attribute store
            p._person_store.cfg.age_gender_interval_sec = CFG.age_gender_interval_sec  # Propagate age/gender interval to the attribute store
            p._person_store.cfg.deepface_emotion_smooth_window = CFG.deepface_emotion_smooth_window  # Propagate smoothing window to the attribute store
            p._async_face.cfg.deepface_frame_skip       = CFG.deepface_frame_skip  # Propagate frame skip to the async DeepFace pool
            p._async_face.cfg.deepface_emotion_confidence_threshold = CFG.deepface_emotion_confidence_threshold  # Propagate confidence threshold to the async DeepFace pool
            p._action_rec.cfg.action_interval_frames    = CFG.action_interval_frames  # Propagate action interval to the action recogniser

    def on_apply(_b):
        """Handler for the 'Apply Config' button click."""
        with status_out:
            clear_output(); _sync()  # Clear previous status output and synchronise all widget values
            _run_console.print("  [bold green]Configuration applied.[/bold green]")  # Confirm that the new configuration is active
    apply_btn.on_click(on_apply)  # Register the button click handler

    def _sec(t):
        """Helper for creating section headers in the dashboard."""
        return widgets.HTML(
        f'<div style="color:#ffaa00;font-family:monospace;margin-top:10px;font-size:13px;">-- {t}</div>')  # Return an HTML widget styled as an amber monospace section header

    header = widgets.HTML(
        '<div style="background:#0d0d0d;border:2px solid #cc0000;padding:14px 18px;'
        'font-family:monospace;color:#ff3333;font-size:16px;border-radius:4px;">'
        ' EyeOfAI - LIVE CONFIGURATION PANEL<br>'
        '<span style="color:#888;font-size:11px;">All changes apply immediately.</span></div>')  # Dashboard header HTML with dark background and red border

    panel = widgets.VBox([
        header,  # Top header with title and subtitle
        _sec("DETECTION"),  # Section header for detection-related controls
        thresh_w, track_w, fps_w,  # Detection threshold, track threshold, and FPS sliders
        _sec("GOVERNOR"),  # Section header for governor controls
        gov_fps_w,  # Governor target FPS slider
        _sec("DEEPFACE"),  # Section header for DeepFace analysis controls
        emotion_w, age_gender_w, df_skip_w, df_emo_smooth_w, df_emo_conf_w,  # All DeepFace-related sliders
        _sec("ACTION RECOGNITION"),  # Section header for action recognition controls
        action_interval_w,  # Action inference interval slider
        _sec("ASYNC"),  # Section header for async worker controls
        workers_w,  # Async workers slider
        _sec("DISPLAY"),  # Section header for display controls
        show_hud_w,  # HUD overlay checkbox
        widgets.HBox([apply_btn]), status_out,  # Apply button row and status output area
    ], layout=widgets.Layout(border="2px solid #330000", padding="16px", width="590px"))  # Wrap all controls in a VBox with a dark red border
    ipy_display(panel)  # Render the dashboard panel in the Colab notebook

_run_console.print("  [bold green]Configuration dashboard defined.[/bold green]")  # Confirm that the dashboard function was defined successfully

# ============================================================ #
# ENTRY POINT
# Handles video upload and pipeline execution in Colab.
# ============================================================ #
from google.colab import files as colab_files  # Import the Colab files API for uploading and downloading files

def upload_and_run(max_frames=300, show_hud=False):
    """
    Prompts user to upload a video, then runs the EyeOfAI pipeline on it.
    Displays results in Colab and offers to download the output video.
    """
    global _active_pipeline  # Declare intent to update the module-level active pipeline reference
    CFG.show_hud = show_hud  # Apply the show_hud argument to the global configuration
    _run_console.print("  [bold blue]Please upload a video file to proceed...[/bold blue]")  # Prompt the user to upload a video
    uploaded = colab_files.upload()  # Open the Colab file upload dialog and wait for the user
    for fname, data in uploaded.items():  # Iterate over uploaded files (usually just one)
        tmp = f"/content/uploaded_{fname}"  # Build a temporary path for the uploaded file
        with open(tmp, "wb") as f: f.write(data)  # Write the uploaded binary data to the temporary file
        _run_console.print(f"  [bold green]Uploaded: {fname} ({len(data)/1e6:.1f} MB)[/bold green]")  # Report the uploaded filename and size

        pipeline         = EyeOfAIPipeline(CFG)  # Create a new pipeline instance with the current global config
        _active_pipeline = pipeline  # Store the pipeline reference for live dashboard updates
        launch_config_dashboard(pipeline)  # Render the interactive configuration dashboard

        output_path = f"/content/eyeofai_{fname}"  # Build the output video path using the uploaded filename
        pipeline.run_colab(source=tmp, max_frames=max_frames, output_path=output_path)  # Run the pipeline on the uploaded video for up to max_frames frames
        pipeline.status()  # Print a full status report after processing completes
        _run_console.print(f"  [bold blue]Downloading processed video: {output_path}[/bold blue]")  # Notify the user that download is starting
        colab_files.download(output_path)  # Trigger the Colab file download for the processed video
        break  # Process only the first uploaded file

if __name__ == "__main__":
    _run_console.print("\n  [bold blue]Starting EyeOfAI pipeline setup. Please follow prompts for video upload.\n[/bold blue]")  # Print the startup message when the script is run directly
    orig = os.getcwd()  # Save the current working directory before switching
    try:
        os.chdir("/content/Detic")  # Switch to the Detic root directory for relative path resolution
        upload_and_run(max_frames=1000)  # Launch the upload-and-run workflow with a 300-frame limit
    except Exception as e:
        _run_console.print(f"[bold red]Critical pipeline error during execution: {e}[/bold red]")  # Report any top-level exception that crashed the pipeline
        MONITOR.log_event(f"Critical pipeline error: {e}", "ERROR")  # Log the critical error in the system monitor
    finally:
        os.chdir(orig)  # Always restore the original working directory on exit